# Import Library and Dependencies

In [ ]:

# Pustaka Standar dan Utilitas
import time
import calendar
import random
import itertools
from datetime import timedelta

# Pemrosesan dan Manipulasi Data
import pandas as pd
import numpy as np

# Ekstraksi Data Web (Web Scraping)
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Visualisasi Data
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Analisis Statistik dan Runtun Waktu
import statsmodels.api as sm
import holidays
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf, ccf
from statsmodels.tsa.seasonal import STL
from scipy.stats import pearsonr, spearmanr


# Pemelajaran Mesin (Machine Learning)
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import KNNImputer

# Algoritma Boosting
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


# Library Save Model
import os
import joblib

# Scraping Data Rainfall

In [ ]:
# def scrape_rainfall_with_calendar_logic(city_name: str = "Surabaya") -> pd.DataFrame:
#     """
#     Melakukan scraping data curah hujan harian berdasarkan tahun dan bulan.

#     Args:
#         city_name (str): Nama kota yang akan dicari datanya. Default adalah 'Surabaya'.

#     Returns:
#         pd.DataFrame: DataFrame yang berisi data tahun, bulan, hari, nilai curah hujan, dan status.
#     """
#     # Konfigurasi WebDriver
#     chrome_options = Options()
#     # chrome_options.add_argument("--headless") # Aktifkan untuk mode tanpa jendela
#     driver = webdriver.Chrome(
#         service=Service(ChromeDriverManager().install()), 
#         options=chrome_options
#     )
    
#     # URL target berdasarkan kota
#     url = f"https://weather-and-climate.com/average-monthly-precipitation-Rainfall,{city_name},Indonesia"
#     all_data = []
    
#     try:
#         driver.get(url)
#         wait = WebDriverWait(driver, 20)
        
#         # Rentang tahun observasi
#         years = [2024, 2025]
        
#         for year in years:
#             for month in range(1, 13):
#                 # Mendapatkan jumlah hari dalam bulan berjalan
#                 _, days_in_month = calendar.monthrange(year, month)
#                 print(f"Memproses {year}-{month:02d} (Target: {days_in_month} hari)")

#                 # Menunggu hingga komponen kalender tersedia
#                 component = wait.until(
#                     EC.presence_of_element_located((By.CLASS_NAME, "weather-history-component"))
#                 )
                
#                 # Memilih tahun dan bulan melalui dropdown
#                 Select(component.find_element(By.CSS_SELECTOR, "select.weather-history-year-select")).select_by_value(str(year))
#                 Select(component.find_element(By.CSS_SELECTOR, "select.weather-history-month-dropdown")).select_by_value(str(month))
                
#                 # Validasi kemunculan hari terakhir untuk memastikan data terunggah sempurna
#                 xpath_last_day = f"//div[contains(@class, 'weather-history-date-num') and text()='{days_in_month}']"
#                 try:
#                     wait.until(EC.presence_of_element_located((By.XPATH, xpath_last_day)))
#                     time.sleep(1) # Jeda singkat untuk rendering nilai curah hujan (mm)
#                 except:
#                     print(f"Peringatan: Tanggal {days_in_month} pada {year}-{month} tidak ditemukan. Dilewati.")
#                     continue

#                 # Parsing konten HTML menggunakan BeautifulSoup
#                 soup = BeautifulSoup(driver.page_source, 'html.parser')
#                 calendar_div = soup.find('div', class_='weather-history-calendar')
#                 day_cells = calendar_div.find_all('div', class_='weather-history-day')
                
#                 for cell in day_cells:
#                     # Lewati sel yang tidak memiliki data tanggal
#                     if 'no-data' in cell.get('class', []):
#                         continue
                        
#                     day_num_text = cell.find('div', class_='weather-history-date-num').get_text(strip=True)
#                     day_num = int(day_num_text)
                    
#                     # Ekstraksi nilai curah hujan (mm)
#                     temp_div = cell.find('div', class_='weather-history-temp')
#                     rainfall_val = float(temp_div.get_text(strip=True).replace(' mm', '')) if temp_div else 0.0
                    
#                     # Ekstraksi status cuaca berdasarkan nama class CSS
#                     status = "None"
#                     for cls in cell.get('class', []):
#                         if cls.startswith('rain-'):
#                             status = cls.replace('rain-', '').capitalize()
                    
#                     # Menyimpan data ke dalam list
#                     all_data.append({
#                         'year': year,
#                         'month': month,
#                         'day': day_num,
#                         'rainfall': rainfall_val,
#                         'status': status,
#                         'date': f"{year}-{month:02d}-{day_num:02d}"
#                     })
                    
#                     # Berhenti jika sudah mencapai hari terakhir dalam bulan tersebut
#                     if day_num == days_in_month:
#                         break 

#         # Konversi hasil ke DataFrame dan ekspor ke CSV
#         df_final = pd.DataFrame(all_data)
#         df_final.to_csv(f'Tambahan/Data_Rainfall_{city_name}_2.csv', index=False)
#         return df_final

#     except Exception as e:
#         print(f"Terjadi kesalahan saat scraping: {str(e)}")
#         return pd.DataFrame()

#     finally:
#         driver.quit()

# # Eksekusi fungsi utama
# df_hujan = scrape_rainfall_with_calendar_logic("Surabaya")

# Read Dataset

In [ ]:
def load_all_datasets():
    """
    Memuat seluruh dataset yang diperlukan untuk analisis klaim dan fitur eksogen.
    
    Returns:
        tuple: Mengembalikan empat DataFrame (klaim, polis, curah hujan Jakarta, curah hujan Surabaya).
    """
    # Jalur folder dataset
    base_path = 'Tambahan/'
    
    # Memuat data transaksi klaim asuransi
    df_klaim = pd.read_csv(f'{base_path}Data_Klaim.csv')
    
    # Memuat data master polis nasabah
    df_polis = pd.read_csv(f'{base_path}Data_Polis.csv')
    
    # Memuat data fitur eksogen curah hujan wilayah Jakarta
    df_rainfall_jkt = pd.read_csv(f'{base_path}Data_Rainfall_Jakarta.csv')
    
    # Memuat data fitur eksogen curah hujan wilayah Surabaya
    df_rainfall_sby = pd.read_csv(f'{base_path}Data_Rainfall_Surabaya.csv')
    
    return df_klaim, df_polis, df_rainfall_jkt, df_rainfall_sby

# Eksekusi pemuatan data ke dalam variabel global
df_klaim, df_polis, df_rainfall_jkt, df_rainfall_sby = load_all_datasets()

# Menampilkan informasi ringkas ukuran dataset untuk validasi awal
print(f"Dataset Klaim   : {df_klaim.shape[0]} baris")
print(f"Dataset Polis   : {df_polis.shape[0]} baris")
print(f"Rainfall Jakarta : {df_rainfall_jkt.shape[0]} baris")
print(f"Rainfall Surabaya: {df_rainfall_sby.shape[0]} baris")

# Preprocessing

## Pivot Data Rainfall

In [ ]:
def aggregate_monthly_rainfall(df_rainfall, city_suffix):
    """
    Mengagregasi data curah hujan harian menjadi metrik statistik bulanan.

    Args:
        df_rainfall (pd.DataFrame): DataFrame curah hujan harian.
        city_suffix (str): Akhiran nama kota untuk penamaan kolom (misal: 'Jkt').

    Returns:
        pd.DataFrame: DataFrame hasil agregasi bulanan.
    """
    # 1. Pastikan kolom tanggal bertipe datetime
    df_rainfall['date'] = pd.to_datetime(df_rainfall['date'])
    
    # 2. Proses Agregasi berdasarkan Tahun dan Bulan
    monthly_pivot = df_rainfall.groupby(['year', 'month']).agg(
        **{f'Total_Rainfall_mm_{city_suffix}': ('rainfall', 'sum')},
        **{f'Avg_Daily_Rainfall_{city_suffix}': ('rainfall', 'mean')},
        **{f'Max_Daily_Rainfall_{city_suffix}': ('rainfall', 'max')},
        # Menghitung jumlah hari dengan curah hujan > 0
        **{f'Rainy_Days_Count_{city_suffix}': ('rainfall', lambda x: (x > 0).sum())},
        # Menghitung hari dengan status intensitas tinggi atau berat
        **{f'Extreme_Rain_Days_{city_suffix}': ('status', lambda x: x.isin(['High', 'Heavy']).sum())}
    ).reset_index()
    
    return monthly_pivot

# --- Eksekusi Agregasi untuk Jakarta ---
monthly_rainfall_jkt = aggregate_monthly_rainfall(df_rainfall_jkt, 'Jkt')
print("Hasil Pivot Rainfall JAKARTA:")
display(monthly_rainfall_jkt.head())
monthly_rainfall_jkt.to_csv('Tambahan/Monthly_Rainfall_Jakarta_Pivot.csv', index=False)

# --- Eksekusi Agregasi untuk Surabaya ---
monthly_rainfall_sby = aggregate_monthly_rainfall(df_rainfall_sby, 'Sby')
print("\nHasil Pivot Rainfall SURABAYA:")
display(monthly_rainfall_sby.head())
monthly_rainfall_sby.to_csv('Tambahan/Monthly_Rainfall_Surabaya_Pivot.csv', index=False)

## Transform Temporal Data

In [ ]:
def preprocess_temporal_data(df_klaim, df_polis):
    """
    Melakukan imputasi nilai kosong dan standarisasi tipe data tanggal pada dataset.

    Args:
        df_klaim (pd.DataFrame): DataFrame transaksi klaim.
        df_polis (pd.DataFrame): DataFrame master polis nasabah.

    Returns:
        tuple: DataFrame klaim dan polis yang telah diproses.
    """
    
    # 1. Imputasi Tanggal Pembayaran Klaim
    # Nilai kosong pada tanggal pembayaran diisi dengan tanggal masuk RS sebagai asumsi proses tercepat
    df_klaim['Tanggal Pembayaran Klaim'] = df_klaim['Tanggal Pembayaran Klaim'].fillna(
        df_klaim['Tanggal Pasien Masuk RS']
    )

    # 2. Konversi Kolom Tanggal pada Dataset Klaim
    # Mengubah kolom target menjadi format datetime standar pandas
    cols_tgl_klaim = ['Tanggal Pembayaran Klaim', 'Tanggal Pasien Masuk RS', 'Tanggal Pasien Keluar RS']
    for col in cols_tgl_klaim:
        df_klaim[col] = pd.to_datetime(df_klaim[col])

    # 3. Konversi Kolom Tanggal pada Dataset Polis
    # Menggunakan format '%Y%m%d' untuk menangani representasi tanggal dalam bentuk integer/string
    df_polis['Tanggal Lahir'] = pd.to_datetime(
        df_polis['Tanggal Lahir'].astype(str), format='%Y%m%d'
    )
    df_polis['Tanggal Efektif Polis'] = pd.to_datetime(
        df_polis['Tanggal Efektif Polis'].astype(str), format='%Y%m%d'
    )

    return df_klaim, df_polis

# Eksekusi fungsi pra-pemrosesan
df_klaim, df_polis = preprocess_temporal_data(df_klaim, df_polis)

# Validasi hasil konversi tipe data
print("Tipe data kolom tanggal pada df_klaim:")
print(df_klaim[['Tanggal Pasien Masuk RS', 'Tanggal Pembayaran Klaim']].dtypes)

## Cek Missing Value

In [ ]:
missing_summary = df_klaim.isnull().sum()
print("Jumlah Nilai Kosong Sebelum Imputasi:")
print(missing_summary)

missing_summary_polis = df_polis.isnull().sum()
print("Jumlah Nilai Kosong Sebelum Imputasi:")
print(missing_summary_polis)

## Impute Missing Value

In [ ]:
def handle_missing_values(df):
    """
    Melakukan imputasi pada kolom kategorikal dan tanggal yang memiliki nilai kosong.

    Args:
        df (pd.DataFrame): DataFrame yang akan diproses.

    Returns:
        pd.DataFrame: DataFrame yang telah dibersihkan dari nilai null.
    """
    # 1. Imputasi Kolom Kategorikal
    # Mengisi nilai kosong dengan label 'Unknown' agar tidak mengganggu proses encoding
    categorical_cols = ['Inpatient/Outpatient', 'Lokasi RS', 'ICD Diagnosis', 'ICD Description']
    for col in categorical_cols:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown')

    return df

# Eksekusi fungsi penanganan nilai kosong
df_klaim = handle_missing_values(df_klaim)

# 3. Validasi Akhir Nilai Kosong
# Mencetak ringkasan jumlah nilai null untuk memastikan dataset sudah bersih
missing_summary = df_klaim.isnull().sum()
print("Jumlah Nilai Kosong Setelah Imputasi:")
print(missing_summary)

## Merge Dataset

In [ ]:
df = pd.merge(df_klaim, df_polis, on='Nomor Polis', how='left')

In [ ]:
df.head(20)

## Convert ICD Diagnosis ke Capital Semua

In [ ]:
df['ICD Diagnosis'] = df['ICD Diagnosis'].astype(str).str.upper()
df['ICD_Group'] = df['ICD Diagnosis'].str[0]

## Outlier Handling

In [ ]:
def process_and_visualize_outliers(df, target_col, feature_col, quantile=0.97):
    """
    Mengidentifikasi, memvisualisasikan, dan melakukan capping pada outlier.

    Args:
        df (pd.DataFrame): DataFrame input.
        target_col (str): Nama kolom target (nominal klaim).
        feature_col (str): Nama kolom fitur (biaya RS).
        quantile (float): Nilai ambang batas kuantil. Default 0.97.

    Returns:
        pd.DataFrame: DataFrame dengan kolom baru 'Nominal_Capped'.
    """
    
    # 1. Identifikasi dan Penentuan Ambang Batas
    # Menghitung nilai batas atas berdasarkan kuantil yang ditentukan
    limit = df[target_col].quantile(quantile)
    df['Status_Outlier'] = df[target_col] > limit
    
    # Menentukan batas y-axis global agar skala visual konsisten (offset 5%)
    y_max = df[target_col].max() * 1.05

    # 2. Visualisasi Kondisi Sebelum Capping
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=df, x=feature_col, y=target_col, 
        hue='Status_Outlier', palette={True: '#e74c3c', False: '#3498db'}, 
        alpha=0.5
    )
    
    plt.axhline(y=limit, color='black', linestyle='--', label=f'Batas Cap (Rp {limit:,.0f})')
    plt.ylim(0, y_max) # Skala dikunci berdasarkan nilai maksimum asli
    plt.title('Identifikasi Outlier (Sebelum Capping - Skala Terkunci)', fontsize=14)
    plt.xlabel('Nominal Biaya Rumah Sakit', fontsize=12)
    plt.ylabel('Nominal Klaim Disetujui', fontsize=12)
    plt.legend(title='Keterangan')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 3. Eksekusi Capping Data
    # Membatasi nilai ekstrem tepat pada ambang batas limit
    df['Nominal_Capped'] = df[target_col].clip(upper=limit)

    # 4. Visualisasi Kondisi Sesudah Capping
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=df, x=feature_col, y='Nominal_Capped', 
        hue='Status_Outlier', palette={True: '#f1c40f', False: '#2ecc71'}, 
        alpha=0.5
    )
    
    plt.axhline(y=limit, color='black', linestyle='--', label='Batas Cap')
    plt.ylim(0, y_max) # Skala tetap menggunakan y_max yang sama untuk komparasi
    plt.title('Hasil Capping (Sesudah Capping - Skala Terkunci)', fontsize=14)
    plt.xlabel('Nominal Biaya Rumah Sakit', fontsize=12)
    plt.ylabel('Nominal Klaim (Ter-Capping)', fontsize=12)
    plt.legend(title='Keterangan', labels=['Normal', 'Hasil Capping'])
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return df

# Eksekusi Fungsi
df = process_and_visualize_outliers(
    df, 
    target_col='Nominal Klaim Yang Disetujui', 
    feature_col='Nominal Biaya RS Yang Terjadi'
)

# Pivot Table Monthly

In [ ]:
# 1. Menentukan periode bulan sebagai dasar pengelompokan
monthly_grouper = df['Tanggal Pasien Masuk RS'].dt.to_period('M')

# 2. Agregasi fitur kategorikal menggunakan Cross-Tabulation
# Mengelompokkan distribusi frekuensi untuk setiap kategori per bulan
monthly_counts = pd.concat([
    pd.crosstab(monthly_grouper, df['Inpatient/Outpatient']).add_prefix('Type_'),
    pd.crosstab(monthly_grouper, df['Domisili']).add_prefix('Dom_'),
    pd.crosstab(monthly_grouper, df['Gender']).add_prefix('Gen_'),
    pd.crosstab(monthly_grouper, df['Lokasi RS']).add_prefix('Loc_'), 
    pd.crosstab(monthly_grouper, df['ICD_Group']).add_prefix('ICD_'),
    pd.crosstab(monthly_grouper, df['Plan Code']).add_prefix('Plan_'),
    pd.crosstab(monthly_grouper, df['Reimburse/Cashless']).add_prefix('Pay_')
], axis=1).fillna(0)

# 3. Agregasi Variabel Target (Total Claim dan Frekuensi)
target_agg = df.groupby(monthly_grouper).agg(
    Total_Claim=('Nominal_Capped', 'sum'),
    Claim_Frequency=('Claim ID', 'count')
)

# Menghitung Claim Severity (Rata-rata biaya per klaim)
target_agg['Claim_Severity'] = target_agg['Total_Claim'] / target_agg['Claim_Frequency']

# 4. Penggabungan Data Agregasi
monthly_data = pd.concat([monthly_counts, target_agg], axis=1).reset_index()
monthly_data.rename(columns={monthly_data.columns[0]: 'YearMonth'}, inplace=True)

# Pivot Table Weekly

In [ ]:
# 1. Menentukan Periode Mingguan
weekly_grouper = df['Tanggal Pasien Masuk RS'].dt.to_period('W')

# 2. Agregasi Fitur Kategorikal (Cross-Tabulation)
# Menghitung frekuensi kemunculan setiap kategori dalam skala mingguan
weekly_counts = pd.concat([
    pd.crosstab(weekly_grouper, df['Inpatient/Outpatient']).add_prefix('Type_'),
    pd.crosstab(weekly_grouper, df['Domisili']).add_prefix('Dom_'),
    pd.crosstab(weekly_grouper, df['Gender']).add_prefix('Gen_'),
    pd.crosstab(weekly_grouper, df['Lokasi RS']).add_prefix('Loc_'),
    pd.crosstab(weekly_grouper, df['ICD_Group']).add_prefix('ICD_'),
    pd.crosstab(weekly_grouper, df['Plan Code']).add_prefix('Plan_'),
    pd.crosstab(weekly_grouper, df['Reimburse/Cashless']).add_prefix('Pay_')
], axis=1).fillna(0)

# 3. Agregasi Variabel Target Mingguan
target_agg_weekly = df.groupby(weekly_grouper).agg(
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum'),
    Claim_Frequency=('Claim ID', 'count')
)

# Menghitung Rata-rata Keparahan Klaim (Severity)
target_agg_weekly['Claim_Severity'] = target_agg_weekly['Total_Claim'] / target_agg_weekly['Claim_Frequency']
target_agg_weekly = target_agg_weekly.fillna(0)

# 4. Penggabungan Dataset Mingguan
weekly_data = pd.concat([weekly_counts, target_agg_weekly], axis=1).reset_index()
weekly_data.rename(columns={weekly_data.columns[0]: 'YearWeek'}, inplace=True)

# Filtering data untuk memastikan format YearWeek yang valid
weekly_data_clean = weekly_data[weekly_data['YearWeek'].astype(str).str.len() > 12].reset_index(drop=True)

# Feature Engineering

## Feature Eksogen

In [ ]:
# Pembentukan Dataset Utama
monthly_data = target_agg.reset_index().rename(columns={'Tanggal Pasien Masuk RS': 'YearMonth'})
monthly_data['Year_Num'] = monthly_data['YearMonth'].dt.year
monthly_data['Month_Num'] = monthly_data['YearMonth'].dt.month

# Integrasi Data Cuaca (Jakarta & Surabaya) secara dinamis
weather_datasets = [(monthly_rainfall_jkt, 'Jkt'), (monthly_rainfall_sby, 'Sby')]
for df_w, suffix in weather_datasets:
    monthly_data = pd.merge(
        monthly_data, df_w, 
        left_on=['Year_Num', 'Month_Num'], 
        right_on=['year', 'month'], 
        how='left'
    ).drop(['year', 'month'], axis=1)

# 2. Encoding Siklik Bulan (Sin/Cos) untuk menangkap pola berulang tahunan
monthly_data['month_sin'] = np.sin(2 * np.pi * monthly_data['Month_Num'] / 12)
monthly_data['month_cos'] = np.cos(2 * np.pi * monthly_data['Month_Num'] / 12)

## Feature Endogen (Time Series)

In [ ]:
# 1. Pembuatan Fitur Lag (1 hingga 4 bulan ke belakang)
for lag in range(1, 5):
    monthly_data[f'Lag_{lag}_Freq'] = monthly_data['Claim_Frequency'].shift(lag)
    monthly_data[f'Lag_{lag}_Total'] = monthly_data['Total_Claim'].shift(lag)
    monthly_data[f'Lag_{lag}_Sev'] = monthly_data['Claim_Severity'].shift(lag)

# 2. Pembuatan Fitur Rolling Mean (RM) untuk tren 2, 3, dan 4 bulan
for window in [2, 3, 4]:
    monthly_data[f'RM_{window}_Freq'] = monthly_data['Claim_Frequency'].shift(1).rolling(window=window).mean()
    monthly_data[f'RM_{window}_Total'] = monthly_data['Total_Claim'].shift(1).rolling(window=window).mean()
    monthly_data[f'RM_{window}_Sev'] = monthly_data['Claim_Severity'].shift(1).rolling(window=window).mean()

# 3. Perhitungan Rolling Standard Deviation untuk volatilitas 
for window in [2, 3, 4]:
    # Volatilitas Frekuensi
    monthly_data[f'RStd_{window}_Freq'] = monthly_data['Claim_Frequency'].shift(1).rolling(window=window).std()
    # Volatilitas Total Klaim
    monthly_data[f'RStd_{window}_Total'] = monthly_data['Total_Claim'].shift(1).rolling(window=window).std()
    # Volatilitas Severity (Rata-rata klaim)
    monthly_data[f'RStd_{window}_Sev'] = monthly_data['Claim_Severity'].shift(1).rolling(window=window).std()

# 4. Perhitungan Selisih (Difference) Periode Sebelumnya
monthly_data['Diff_1_Freq'] = monthly_data['Claim_Frequency'].shift(1) - monthly_data['Claim_Frequency'].shift(2)
monthly_data['Diff_1_Total'] = monthly_data['Total_Claim'].shift(1) - monthly_data['Total_Claim'].shift(2)
monthly_data['Diff_1_Sev'] = monthly_data['Claim_Severity'].shift(1) - monthly_data['Claim_Severity'].shift(2)

# Growth Rate: Rasio pertumbuhan frekuensi klaim
monthly_data['Growth_Rate_Freq'] = (monthly_data['Diff_1_Freq']) / (monthly_data['Lag_2_Freq'] + 1)

# Z-Score untuk mendeteksi anomali frekuensi relatif terhadap tren 3 bulan
monthly_data['Z_Score_Freq'] = (monthly_data['Claim_Frequency'].shift(1) - monthly_data['RM_3_Freq']) / \
                               (monthly_data['RStd_3_Freq'] + 1e-6)

# Fitur akselerasi frekuensi (perubahan dari perubahan)
monthly_data['Lag_1_Diff_1_Freq'] = monthly_data['Claim_Frequency'].shift(2) - monthly_data['Claim_Frequency'].shift(3)
monthly_data['Acc_Freq'] = monthly_data['Diff_1_Freq'] - monthly_data['Lag_1_Diff_1_Freq']

## Fitur Calendar

In [ ]:
# Inisialisasi daftar hari libur Indonesia
id_holidays = holidays.Indonesia(years=[2023, 2024, 2025])

def count_public_holidays(year, month):
    """Menghitung jumlah hari libur nasional dalam satu bulan tertentu."""
    return sum(1 for date in id_holidays if date.year == year and date.month == month)

def count_working_days(year, month):
    """Menghitung jumlah hari kerja (Senin-Jumat) dikurangi hari libur nasional."""
    num_days = calendar.monthrange(year, month)[1]
    start_date = f"{year}-{month:02d}-01"
    # Menghitung hari kerja antara tanggal 1 sampai akhir bulan
    bus_days = np.busday_count(start_date, f"{year}-{month:02d}-{num_days:02d}")
    
    # Cek apakah ada hari libur yang jatuh pada hari kerja (Senin-Jumat)
    hol_weekdays = sum(1 for d in id_holidays if d.year == year and d.month == month and d.weekday() < 5)
    return bus_days - hol_weekdays

# Implementasi fitur kalender ke dataset
monthly_data['Days_in_Month'] = monthly_data.apply(lambda r: calendar.monthrange(int(r['Year_Num']), int(r['Month_Num']))[1], axis=1)
monthly_data['Public_Holidays'] = monthly_data.apply(lambda r: count_public_holidays(int(r['Year_Num']), int(r['Month_Num'])), axis=1)
monthly_data['Working_Days'] = monthly_data.apply(lambda r: count_working_days(int(r['Year_Num']), int(r['Month_Num'])), axis=1)

## Feature Indikator Tren Temporal (Time Index)

In [ ]:
# 1. Pembuatan Indeks Waktu Linier
# Menghasilkan angka urut 0, 1, 2, ... sesuai jumlah baris data
monthly_data['Time_Index'] = np.arange(len(monthly_data))

# 2. Pembuatan Indeks Waktu Kuadratik
# Digunakan untuk menangkap percepatan atau perlambatan tren (non-linier)
monthly_data['Time_Index_Squared'] = monthly_data['Time_Index'] ** 2

# 3. Fitur Kalender Tambahan
# Mengidentifikasi kuartal dan periode akhir tahun
monthly_data['Quarter'] = ((monthly_data['Month_Num'] - 1) // 3) + 1
monthly_data['Is_Year_End'] = monthly_data['Month_Num'].isin([11, 12]).astype(int)

# Tampilkan cuplikan data untuk validasi indeks
display(monthly_data[['YearMonth', 'Time_Index', 'Time_Index_Squared']].head())

## Feature Indikator Teknis dan Interaksi Antar Fitur

In [ ]:
# 1. Indikator Teknis (Momentum, MACD, dan Stochastic)
monthly_data['MACD_Freq'] = monthly_data['RM_2_Freq'] - monthly_data['RM_4_Freq']
monthly_data['Momentum_2_Freq'] = monthly_data['Lag_1_Freq'] - monthly_data['Lag_2_Freq']

# Stochastic Oscillator untuk frekuensi klaim
monthly_data['Min_3_Freq'] = monthly_data['Claim_Frequency'].shift(1).rolling(window=3).min()
monthly_data['Max_3_Freq'] = monthly_data['Claim_Frequency'].shift(1).rolling(window=3).max()
monthly_data['Stoch_K_Freq'] = (monthly_data['Claim_Frequency'].shift(1) - monthly_data['Min_3_Freq']) / \
                                (monthly_data['Max_3_Freq'] - monthly_data['Min_3_Freq'] + 1e-6)
fitness = pd.read_csv('Tambahan/prediksi_2025.csv')


# 2. Interaksi Fitur Eksogen: Curah Hujan x Klaim Historis
monthly_data['Rain_x_LagFreq_Jkt'] = monthly_data['Avg_Daily_Rainfall_Jkt'] * monthly_data['Lag_1_Freq']
monthly_data['Rain_x_LagFreq_Sby'] = monthly_data['Avg_Daily_Rainfall_Sby'] * monthly_data['Lag_1_Freq']

# 3. Pembersihan Akhir: Menghapus baris dengan nilai kosong akibat shift/rolling
monthly_data_clean = monthly_data.dropna().reset_index(drop=True)


In [ ]:
monthly_data_clean.head(10)


# EDA

In [ ]:
df_klaim.info()

In [ ]:
df_polis.info()

In [ ]:
df_polis.head()

In [ ]:
df_klaim.head()

In [ ]:
missing_value_polis = df_polis.isnull().sum()

print(missing_value_polis)

In [ ]:
missing_value_klaim = df_klaim.isnull().sum()

print(missing_value_klaim)

## Visualisasi Total Claim Per Month Without Cap

In [ ]:
def prepare_annual_data(df):
    """
    Menyiapkan dataset agregasi klaim berdasarkan tahun dan bulan.

    Args:
        df (pd.DataFrame): Dataset transaksi klaim asli.

    Returns:
        pd.DataFrame: DataFrame yang telah di-pivot dan diurutkan secara kronologis.
    """
    # 1. Agregasi data berdasarkan Tahun dan Bulan
    # Menggunakan kolom Tanggal Pasien Masuk RS sebagai dasar waktu
    target_agg = df.groupby([
        df['Tanggal Pasien Masuk RS'].dt.year.rename('Tahun'),
        df['Tanggal Pasien Masuk RS'].dt.month.rename('Bulan')
    ]).agg(
        Total_Claim_Raw=('Nominal Klaim Yang Disetujui', 'sum'),
        Claim_Frequency=('Claim ID', 'count')
    ).reset_index()

    # 2. Pemetaan Nama Bulan
    month_names = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'Mei', 6: 'Jun',
                   7: 'Jul', 8: 'Ags', 9: 'Sep', 10: 'Okt', 11: 'Nov', 12: 'Des'}
    target_agg['Nama Bulan'] = target_agg['Bulan'].map(month_names)

    # 3. Standarisasi Urutan Kategorikal
    target_agg['Bulan'] = pd.Categorical(target_agg['Bulan'], categories=range(1, 13), ordered=True)
    
    return target_agg.sort_values(['Tahun', 'Bulan'])


def plot_annual_comparison(data):
    """
    Menghasilkan visualisasi perbandingan tren klaim tahunan.

    Args:
        data (pd.DataFrame): Data agregasi tahunan.
    """
    plt.figure(figsize=(15, 8))
    sns.set_theme(style="whitegrid")

    # Definisi palet warna standar (Biru: 2024, Merah: 2025)
    colors = {2024: '#3498db', 2025: '#e74c3c'}

    # 1. Pembuatan Grafik Garis
    sns.lineplot(
        data=data, x='Nama Bulan', y='Total_Claim_Raw',
        hue='Tahun', style='Tahun', markers=True, 
        dashes=False, palette=colors, linewidth=3, markersize=10
    )

    # 2. Penambahan Label Nominal pada Setiap Titik
    for year in data['Tahun'].unique():
        subset = data[data['Tahun'] == year]
        y_max = data['Total_Claim_Raw'].max()
        
        # Penentuan posisi label agar tidak tumpang tindih
        offset = (y_max * 0.03) if year == 2024 else -(y_max * 0.06)
        va_align = 'bottom' if year == 2024 else 'top'

        for i in range(len(subset)):
            plt.text(
                x=subset['Nama Bulan'].iloc[i],
                y=subset['Total_Claim_Raw'].iloc[i] + offset,
                s=f"Rp {subset['Total_Claim_Raw'].iloc[i]/1e6:.1f}M",
                ha='center', va=va_align, fontsize=9, 
                fontweight='bold', color=colors[year]
            )

    # 3. Konfigurasi Estetika Akhir
    plt.title('Perbandingan Tren Akumulasi Nominal Klaim Bulanan: 2024 vs 2025', fontsize=14)
    plt.xlabel('Bulan', fontsize=12)
    plt.ylabel('Total Nominal Klaim (Rp)', fontsize=12)
    plt.legend(title='Tahun', loc='upper right')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# Eksekusi persiapan data
target_agg_year = prepare_annual_data(df)

# Eksekusi visualisasi
plot_annual_comparison(target_agg_year)

## Visualisasi Distribusi Domisili Polis

In [ ]:
# 1. Penghitungan Frekuensi Pasien Berdasarkan Domisili
# Mengambil 15 wilayah dengan jumlah pasien terbanyak
domisili_counts = df['Domisili'].value_counts().reset_index()
domisili_counts.columns = ['Domisili', 'Jumlah_Pasien']
top_15_domisili = domisili_counts.head(15)

# 2. Visualisasi Grafik Batang Horizontal
plt.figure(figsize=(12, 7))

# Menggunakan orientasi horizontal untuk keterbacaan label wilayah
sns.barplot(
    data=top_15_domisili, 
    x='Jumlah_Pasien', 
    y='Domisili', 
    palette='viridis'
)

# Pengaturan Judul dan Label Sumbu
plt.title('Top 15 Domisili Pasien Berdasarkan Frekuensi Klaim', fontsize=14, fontweight='bold')
plt.xlabel('Jumlah Pasien (Frekuensi Klaim)', fontsize=12)
plt.ylabel('Wilayah Domisili', fontsize=12)

# Menambahkan garis bantu vertikal untuk mempermudah pembacaan skala
plt.grid(axis='x', linestyle='--', alpha=0.3)

# Optimasi Tata Letak
plt.tight_layout()
plt.show()

# 3. Menampilkan Ringkasan Data dalam Bentuk Tabel
print("Ringkasan 15 Wilayah Domisili Teratas:")
display(top_15_domisili)

## Visualisasi Tren Target Feature Harian

In [ ]:
def aggregate_daily_claims(df):
    """
    Mengagregasi data transaksi klaim menjadi ringkasan statistik harian.

    Args:
        df (pd.DataFrame): Dataset transaksi klaim asli.

    Returns:
        pd.DataFrame: DataFrame dengan agregasi harian (Frequency, Total, Severity).
    """
    # 1. Agregasi harian menggunakan kolom tanggal masuk
    # Mengelompokkan berdasarkan tanggal murni (date) tanpa informasi jam
    df_daily = df.groupby(df['Tanggal Pasien Masuk RS'].dt.date).agg(
        Claim_Frequency=('Claim ID', 'count'),
        Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
    ).reset_index()

    # 2. Perhitungan Claim Severity harian
    # Menghitung rata-rata biaya per kejadian klaim dalam satu hari
    df_daily['Claim_Severity'] = df_daily['Total_Claim'] / df_daily['Claim_Frequency']
    
    # Memastikan kolom tanggal kembali ke format datetime untuk keperluan visualisasi
    df_daily['Tanggal Pasien Masuk RS'] = pd.to_datetime(df_daily['Tanggal Pasien Masuk RS'])
    
    return df_daily

def plot_daily_performance_trends(df_daily):
    """
    Menampilkan grafik panel tren harian untuk frekuensi, total klaim, dan severity.

    Args:
        df_daily (pd.DataFrame): Dataset agregasi harian.
    """
    # 1. Inisialisasi Kanvas Visualisasi (3 Baris, 1 Kolom)
    fig, axes = plt.subplots(3, 1, figsize=(15, 15), sharex=True)
    sns.set_theme(style="whitegrid")

    # Konfigurasi plotting (Kolom, Warna, Judul, Label Y)
    metrics = [
        ('Claim_Frequency', 'blue', 'Tren Harian: Claim Frequency (Jumlah Pasien)', 'Jumlah Pasien'),
        ('Total_Claim', 'red', 'Tren Harian: Total Claim (Nominal Biaya Rp)', 'Nominal (Rupiah)'),
        ('Claim_Severity', 'green', 'Tren Harian: Claim Severity (Rata-rata Biaya)', 'Severity (Rupiah)')
    ]

    # 2. Iterasi Pembuatan Subplot
    for i, (col, color, title, ylabel) in enumerate(metrics):
        sns.lineplot(
            data=df_daily, x='Tanggal Pasien Masuk RS', y=col, 
            ax=axes[i], color=color, marker='o', markersize=4
        )
        axes[i].set_title(title, fontweight='bold', fontsize=12)
        axes[i].set_ylabel(ylabel, fontsize=10)
        
        # Format angka rupiah menggunakan pemisah ribuan (comma)
        if i > 0: # Hanya untuk Total_Claim dan Claim_Severity
            axes[i].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: format(int(x), ',')))
        
        # Pengaturan grid dan limit sumbu waktu
        axes[i].grid(True, alpha=0.3)
        axes[i].xaxis.set_major_locator(mdates.MonthLocator())
        axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

    # 3. Finalisasi Tata Letak
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


# Eksekusi agregasi harian
df_daily = aggregate_daily_claims(df)


# Eksekusi visualisasi multi-panel
plot_daily_performance_trends(df_daily)

## Visualisasi Perbandingan Std dari Target Feature Data Monthly

In [ ]:
def get_normalized_variability(df, features):
    """
    Menghitung standar deviasi dari data yang telah dinormalisasi dengan teknik Min-Max.

    Args:
        df (pd.DataFrame): Dataset yang berisi variabel target.
        features (list): Daftar nama kolom yang akan dianalisis.

    Returns:
        pd.Series: Nilai standar deviasi dari setiap fitur yang telah dinormalisasi.
    """
    # 1. Ekstraksi data dan penerapan rumus Min-Max (x - min) / (max - min)
    df_subset = df[features].copy()
    df_norm = (df_subset - df_subset.min()) / (df_subset.max() - df_subset.min())

    # 2. Perhitungan standar deviasi pada skala 0-1
    return df_norm.std()

# Menentukan daftar fitur target
target_features = ['Total_Claim', 'Claim_Frequency', 'Claim_Severity']

# Eksekusi perhitungan variabilitas
std_results = get_normalized_variability(monthly_data_clean, target_features)

def plot_variability_analysis(std_norm_results):
    """
    Memvisualisasikan hasil perbandingan standar deviasi ternormalisasi.

    Args:
        std_norm_results (pd.Series): Hasil perhitungan standar deviasi.
    """
    # 1. Konfigurasi label dan kanvas
    labels = [f.replace('_', ' ') for f in std_norm_results.index]
    plt.figure(figsize=(10, 6))
    
    # 2. Pembuatan grafik batang
    bars = plt.bar(labels, std_norm_results, color='mediumpurple', alpha=0.8)

    # Menambahkan anotasi nilai presisi di atas setiap batang
    for i, v in enumerate(std_norm_results):
        plt.text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold', fontsize=10)

    # 3. Pengaturan Tipografi (Judul: 14, Label: 12)
    plt.title('Perbandingan Variabilitas (Min-Max Normalized Std Dev)', fontsize=14, fontweight='bold')
    plt.ylabel('Standar Deviasi (Skala 0-1)', fontsize=12)
    plt.xlabel('Variabel Target', fontsize=12)
    
    # Pengaturan estetika tambahan
    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

# Eksekusi visualisasi
plot_variability_analysis(std_results)

## Visualisasi Perbandingan CV dari Target Feature Data Monthly

In [ ]:

def calculate_target_cv(df, features):
    """
    Menghitung ringkasan statistik dan Koefisien Variasi untuk variabel target.

    Args:
        df (pd.DataFrame): Dataset bulanan yang telah dibersihkan.
        features (list): Daftar kolom target yang akan dianalisis.

    Returns:
        pd.DataFrame: DataFrame berisi mean, std, dan CV dalam persen.
    """
    # Melakukan agregasi mean dan standar deviasi
    stats = df[features].agg(['mean', 'std']).T
    
    # Menghitung CV dengan rumus: (Standar Deviasi / Rata-rata) * 100
    stats['CV_Percent'] = (stats['std'] / stats['mean']) * 100
    
    return stats

# Eksekusi fungsi kalkulasi
cv_summary = calculate_target_cv(monthly_data_clean, target_features)
display(cv_summary)

In [ ]:
def plot_cv_analysis(stats_df):
    """
    Membuat visualisasi Bar Chart untuk perbandingan nilai Koefisien Variasi.
    """
    # Menyiapkan label dengan menghapus underscore agar lebih bersih
    labels = [idx.replace('_', ' ') for idx in stats_df.index]
    
    plt.figure(figsize=(10, 6))
    
    # Membuat grafik batang dengan warna kontras
    cv_bars = plt.bar(labels, stats_df['CV_Percent'], color='darkorange', alpha=0.8)

    # Menambahkan anotasi nilai persentase tepat di atas setiap batang
    for bar in cv_bars:
        y_val = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2, 
            y_val + 0.5, 
            f'{y_val:.2f}%', 
            ha='center', 
            fontweight='bold', 
            fontsize=10
        )

    # Pengaturan atribut grafik sesuai standar laporan
    plt.title('Perbandingan Koefisien Variasi (CV) Variabel Target', fontsize=14, fontweight='bold')
    plt.ylabel('Nilai CV (%)', fontsize=12)
    plt.xlabel('Variabel Target', fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Eksekusi fungsi visualisasi
plot_cv_analysis(cv_summary)

## Visualisasi Perbandingan Group By Daily, Weekly, Monthly Signal to Noise Ratio 

In [ ]:
def calculate_snr(df, target_cols):
    """
    Menghitung Signal-to-Noise Ratio (SNR) untuk kolom target tertentu.
    SNR didefinisikan sebagai rasio antara rata-rata (mean) dan standar deviasi.

    Args:
        df (pd.DataFrame): Dataset yang akan dianalisis.
        target_cols (list): Daftar nama kolom target.

    Returns:
        pd.Series: Nilai SNR untuk setiap kolom target.
    """
    # Agregasi statistik dasar
    stats = df[target_cols].agg(['mean', 'std']).T
    
    # Kalkulasi rasio sinyal terhadap gangguan
    return stats['mean'] / stats['std']

# Eksekusi kalkulasi untuk resolusi harian, mingguan, dan bulanan
snr_daily = calculate_snr(df_daily, target_features)
snr_weekly = calculate_snr(weekly_data_clean, target_features)
snr_monthly = calculate_snr(monthly_data_clean, target_features)

# Persiapan label dan posisi sumbu X
labels = [t.replace('_', ' ') for t in target_features]
x = np.arange(len(labels))
width = 0.25  # Lebar batang grafik

plt.figure(figsize=(12, 7))

# Pembuatan batang untuk masing-masing resolusi temporal
plt.bar(x - width, snr_daily, width, label='Harian (Daily)', color='lightcoral')
plt.bar(x, snr_weekly, width, label='Mingguan (Weekly)', color='khaki')
plt.bar(x + width, snr_monthly, width, label='Bulanan (Monthly)', color='mediumseagreen')

# Pengaturan tipografi dan estetika sesuai standar laporan
plt.title('Analisis Stabilitas Sinyal: Perbandingan SNR Lintas Resolusi', fontsize=14, fontweight='bold')
plt.ylabel('Signal-to-Noise Ratio (Mean/Std)', fontsize=12)
plt.xlabel('Variabel Target', fontsize=12)

# Konfigurasi sumbu dan legenda
plt.xticks(x, labels, fontsize=11)
plt.legend(title='Resolusi Temporal', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

## Tabel Perbandingan Average Freq Group By Day

In [ ]:
# Menambahkan kolom nama hari ke dalam dataframe
df_daily['Hari'] = df_daily['Tanggal Pasien Masuk RS'].dt.day_name()

# Menghitung rata-rata frekuensi klaim dan mengurutkan sesuai urutan hari
tabel_rata_rata_hari = df_daily.groupby('Hari')['Claim_Frequency'].mean().reindex([
    'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'
]).reset_index()

# Penyesuaian nama kolom untuk pelaporan
tabel_rata_rata_hari.columns = ['Hari', 'Rata-rata Claim Frequency']

tabel_rata_rata_hari

## Visualisasi Perbandingan Average Freq Group By Day

In [ ]:
# Menyiapkan urutan hari dan translasi ke Bahasa Indonesia
hari_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
mapping_hari = {
    'Monday': 'Senin', 'Tuesday': 'Selasa', 'Wednesday': 'Rabu',
    'Thursday': 'Kamis', 'Friday': 'Jumat', 'Saturday': 'Sabtu', 'Sunday': 'Minggu'
}

# 1. Menambahkan kolom nama hari
df_daily['Hari'] = df_daily['Tanggal Pasien Masuk RS'].dt.day_name()

# 2. Agregasi rata-rata Claim Frequency per hari
df_avg_hari = df_daily.groupby('Hari')['Claim_Frequency'].mean().reindex(hari_order).reset_index()

# 3. Translasi nama hari untuk visualisasi
df_avg_hari['Hari_Indo'] = df_avg_hari['Hari'].map(mapping_hari)

# 4. Membuat Bar Plot
plt.style.use('seaborn-v0_8-whitegrid')
plt.bar(df_avg_hari['Hari_Indo'], df_avg_hari['Claim_Frequency'], color='royalblue', alpha=0.8)

# Pengaturan label dan judul sesuai standar laporan
plt.title('Rata-rata Claim Frequency Berdasarkan Hari', fontsize=14, fontweight='bold')
plt.xlabel('Hari', fontsize=12)
plt.ylabel('Rata-rata Jumlah Klaim', fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# Menyimpan hasil plot dan data
plt.tight_layout()

## Tabel Distribusi Feature

In [ ]:
icd_group_counts = df['ICD Diagnosis'].astype(str).str[0].value_counts()

print("Distribusi Kelompok Penyakit (Huruf Depan ICD):")
print(icd_group_counts)

In [ ]:
location_counts = df['Lokasi RS'].value_counts(dropna=False)

print(location_counts)


In [ ]:
gender_counts = df['Gender'].value_counts(dropna=False)

print(gender_counts)


In [ ]:
domisili_counts = df['Domisili'].value_counts(dropna=False)

print(domisili_counts)

In [ ]:
status_rawat_counts = df['Inpatient/Outpatient'].value_counts(dropna=False)

print(status_rawat_counts)


In [ ]:
payment_counts = df['Reimburse/Cashless'].value_counts(dropna=False)

print(payment_counts)

In [ ]:
plan_code_counts = df['Plan Code'].value_counts(dropna=False)

print(plan_code_counts)

## Tabel Korelasi Antar Fitur Target Group By Month

In [ ]:
# Menghitung matriks korelasi tanpa nilai absolut
target_correlation = monthly_data_clean[target_features].corr()

print("MATRIKS KORELASI ANTAR TARGET")
display(target_correlation)

Claim_Severity = Total_Claim/Claim_Frequency

## Tabel Korelasi Fitur prediksi dengan Fitur Target

In [ ]:
# 1. Definisi daftar fitur lengkap
ALL_FEATURES = [
    'Month_Num', 'month_sin', 'month_cos', 'Lag_1_Sev', 'Lag_1_Total', 'Lag_1_Freq', 
    'Lag_2_Sev', 'Lag_2_Total', 'Lag_2_Freq', 'Lag_3_Sev', 'Lag_3_Total', 'Lag_3_Freq', 
    'Lag_4_Sev', 'Lag_4_Total', 'Lag_4_Freq', 'Total_Rainfall_mm_Jkt', 'Avg_Daily_Rainfall_Jkt', 
    'Max_Daily_Rainfall_Jkt', 'Rainy_Days_Count_Jkt', 'Extreme_Rain_Days_Jkt', 
    'Total_Rainfall_mm_Sby', 'Avg_Daily_Rainfall_Sby', 'Max_Daily_Rainfall_Sby', 
    'Rainy_Days_Count_Sby', 'Extreme_Rain_Days_Sby', 'RM_2_Freq', 'RM_2_Total', 'RM_2_Sev', 
    'RM_3_Total', 'RM_3_Freq', 'RM_3_Sev', 'RM_4_Freq', 'RM_4_Total', 'RM_4_Sev', 
    'RStd_2_Freq', 'RStd_2_Total', 'RStd_2_Sev', 'RStd_3_Total', 'RStd_3_Freq', 
    'RStd_3_Sev', 'RStd_4_Freq', 'RStd_4_Total', 'RStd_4_Sev', 'Time_Index', 
    'Time_Index_Squared', 'Quarter', 'Is_Year_End', 'Diff_1_Total', 'Diff_1_Freq', 
    'Diff_1_Sev', 'Z_Score_Freq', 'Lag_1_Diff_1_Freq', 'Acc_Freq', 'Min_3_Freq', 
    'Max_3_Freq', 'Stoch_K_Freq', 'MACD_Freq', 'Days_in_Month', 'Public_Holidays', 
    'Working_Days', 'Momentum_2_Freq', 'Growth_Rate_Freq', 'Rain_x_LagFreq_Jkt', 
    'Rain_x_LagFreq_Sby'
]

# 2. Filter data hanya untuk fitur di ALL_FEATURES dan target_features
# Pastikan fitur benar-benar ada dalam dataset untuk menghindari error
available_cols = [col for col in ALL_FEATURES + target_features if col in monthly_data_clean.columns]
df_sub = monthly_data_clean[available_cols]

# 3. Hitung matriks korelasi
correlation_matrix = df_sub.corr()

# 4. Ambil korelasi terhadap target dan urutkan berdasarkan nilai absolut
target_corr = correlation_matrix[target_features].abs()
target_corr = target_corr.drop(index=target_features, errors='ignore')

for target in target_features:
    print(f"\nTOP 20 FEATURES FOR: {target}")
    
    # Mengambil top 20 fitur dengan korelasi tertinggi
    top_feature = target_corr[target].sort_values(ascending=False).head(20)
    
    df_top_feature = pd.DataFrame({
        'Feature': top_feature.index,
        'Correlation_Abs': top_feature.values
    }).reset_index(drop=True)
    
    display(df_top_feature)

## EDA Target Feature

### Visualisasi ACF & PACF

In [ ]:
# 1. Plot ACF & PACF Total_Claim
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
plot_acf(monthly_data['Total_Claim'], ax=ax1, lags=7, title='ACF - Total Claim')
# Menambahkan method='yw' untuk stabilitas pada data kecil
plot_pacf(monthly_data['Total_Claim'], ax=ax2, lags=7, title='PACF - Total Claim', method='yw')
plt.tight_layout()
plt.show()

# 2. Plot ACF & PACF Claim_Frequency
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
plot_acf(monthly_data['Claim_Frequency'], ax=ax1, lags=7, title='ACF - Claim Frequency')
# Menambahkan method='yw'
plot_pacf(monthly_data['Claim_Frequency'], ax=ax2, lags=7, title='PACF - Claim Frequency', color='orange', method='yw')
plt.tight_layout()
plt.show()

# 3. Plot ACF & PACF Claim_Severity
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
plot_acf(monthly_data['Claim_Severity'], ax=ax1, lags=7, title='ACF - Claim_Severity')
plot_pacf(monthly_data['Claim_Severity'], ax=ax2, lags=7, title='PACF - Claim_Severity', color='orange', method='yw')
plt.tight_layout()
plt.show()

### Visualisasi CCF 

In [ ]:

all_pairs = list(itertools.permutations(target_features, 2))
lags = 7
conf_level = 2 / np.sqrt(len(monthly_data_clean))

# Iterasi untuk pembuatan grafik mandiri pada IDE
for target_name, predictor_name in all_pairs:
    # Perhitungan nilai korelasi silang (CCF)
    ccf_vals = sm.tsa.stattools.ccf(
        monthly_data_clean[target_name], 
        monthly_data_clean[predictor_name], 
        adjusted=False
    )[:lags+1]
    
    # Penyesuaian label untuk judul yang deskriptif
    nama_target = target_name.replace('_', ' ')
    nama_prediktor = predictor_name.replace('_', ' ')
    
    # Inisialisasi figur baru untuk setiap skenario
    plt.figure(figsize=(10, 5))
    plt.stem(range(lags + 1), ccf_vals)
    
    
    # Atribut grafik dengan judul deskriptif
    plt.title(f'Lag Predictor {nama_prediktor} Terhadap {nama_target}', fontsize=14, fontweight='bold')
    plt.ylabel('Korelasi', fontsize=12)
    plt.xlabel('Lag (Bulan)', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## EDA Exogeneous Feature

### Month Cos

In [ ]:
monthly_stats = monthly_data.groupby('Month_Num')['Claim_Severity'].agg([
    'mean', 'median', 'std', 'min', 'max', 'count'
]).reset_index()

# 2. Pembulatan nilai untuk kebersihan laporan
monthly_stats = monthly_stats.round(2)

# 3. Menampilkan tabel statistik
print("Tabel Ringkasan Statistik Claim Severity per Bulan:")
display(monthly_stats)


In [ ]:
months = np.arange(1, 13)
cos_values = np.round(np.cos(2 * np.pi * months / 12), 3)

# 2. Membuat tabel pengecekan duplikasi
df_check = pd.DataFrame({'Bulan': months, 'Cos_Value': cos_values})
duplicates = df_check[df_check.duplicated('Cos_Value', keep=False)]

print("Bulan-bulan dengan nilai fitur yang dianggap identik oleh model:")
print(duplicates.sort_values('Cos_Value'))

In [ ]:
# 1. Menyiapkan data kurva dan titik bulan
x_curve = np.linspace(0, 13, 100)
y_curve = np.cos(2 * np.pi * x_curve / 12)
months = np.arange(1, 13)
cos_values = np.cos(2 * np.pi * months / 12)

# 2. Plotting
plt.figure(figsize=(12, 6))
plt.plot(x_curve, y_curve, color='gray', linestyle='--', alpha=0.5, label='Kurva Kosinus')
plt.scatter(months, cos_values, color='red', s=100, zorder=5)

# Menambahkan label bulan dan garis penghubung simetri
for m, v in zip(months, cos_values):
    plt.annotate(str(m), (m, v), textcoords="offset points", xytext=(0,10), ha='center', fontsize=12)

# Menyoroti simetri dengan garis horizontal
pairs = [(1, 11), (2, 10), (3, 9), (4, 8), (5, 7)]
for m1, m2 in pairs:
    y_val = np.cos(2 * np.pi * m1 / 12)
    plt.hlines(y=y_val, xmin=min(m1, m2), xmax=max(m1, m2), color='blue', linestyle=':', alpha=0.6)

plt.title('Visualisasi Simetri month_cos: Identifikasi Tabrakan Nilai Antar Bulan', fontsize=14, fontweight='bold')
plt.xlabel('Bulan (1-12)', fontsize=12)
plt.ylabel('Nilai month_cos', fontsize=12)
plt.axhline(0, color='black', linewidth=0.8)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
pasangan = [(1, 11), (2, 10), (3, 9), (4, 8), (5, 7)]
hasil_perbandingan = []

for m1, m2 in pasangan:
    mean1 = monthly_data[monthly_data['Month_Num'] == m1]['Claim_Severity'].mean()
    mean2 = monthly_data[monthly_data['Month_Num'] == m2]['Claim_Severity'].mean()
    selisih_persen = abs(mean1 - mean2) / ((mean1 + mean2) / 2) * 100
    
    hasil_perbandingan.append({
        'Pasangan_Bulan': f"{m1} & {m2}",
        'Mean_M1': round(mean1, 2),
        'Mean_M2': round(mean2, 2),
        'Selisih_Persen': round(selisih_persen, 2)
    })

# 2. Menampilkan tabel validasi simetri
df_validasi = pd.DataFrame(hasil_perbandingan)
print("Validasi Objektif Keserupaan Pasangan Bulan:")
display(df_validasi)

In [ ]:

ringkasan_data = []

# 2. Iterasi untuk mengambil data secara dinamis dari dataframe
for m1, m2 in pasangan:
    mean_m1 = monthly_data[monthly_data['Month_Num'] == m1]['Claim_Severity'].mean()
    mean_m2 = monthly_data[monthly_data['Month_Num'] == m2]['Claim_Severity'].mean()
    selisih = abs(mean_m1 - mean_m2) / ((mean_m1 + mean_m2) / 2) * 100
    
    ringkasan_data.append({
        'Label': f"{m1} & {m2}",
        'M1': mean_m1,
        'M2': mean_m2,
        'Diff': round(selisih, 2)
    })

df_plot = pd.DataFrame(ringkasan_data)

# 3. Plotting menggunakan data hasil kalkulasi dinamis
x = np.arange(len(df_plot))
width = 0.35
fig, ax = plt.subplots(figsize=(12, 7))

ax.bar(x - width/2, df_plot['M1'], width, label='Bulan Awal (M1)', color='steelblue')
ax.bar(x + width/2, df_plot['M2'], width, label='Bulan Akhir (M2)', color='lightcoral')

# Menambahkan anotasi selisih di atas bar
for i, d in enumerate(df_plot['Diff']):
    y_pos = max(df_plot['M1'][i], df_plot['M2'][i])
    ax.text(i, y_pos + (y_pos * 0.02), f'Selisih: {d}%', ha='center', fontweight='bold')

ax.set_title('Validasi Dinamis Simetri month_cos terhadap Claim Severity', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(df_plot['Label'])
ax.set_ylabel('Mean Claim Severity (IDR)', fontsize=12)
ax.legend()
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

### Max Rainfalll Jakarta

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 7))

# 2. Plotting Claim Frequency (Sumbu Y kiri)
color_freq = 'tab:blue'
ax1.set_xlabel('Indeks Waktu (Bulan)', fontsize=12)
ax1.set_ylabel('Claim Frequency', color=color_freq, fontsize=12)
ax1.plot(monthly_data['Time_Index'], monthly_data['Claim_Frequency'], 
         color=color_freq, marker='o', linewidth=2, label='Claim Frequency')
ax1.tick_params(axis='y', labelcolor=color_freq)

# 3. Plotting Max Rainfall (Sumbu Y kanan)
ax2 = ax1.twinx()
color_rain = 'tab:red'
ax2.set_ylabel('Max Daily Rainfall Jkt (mm)', color=color_rain, fontsize=12)
ax2.plot(monthly_data['Time_Index'], monthly_data['Max_Daily_Rainfall_Jkt'], 
         color=color_rain, linestyle='--', marker='x', alpha=0.7, label='Max Rainfall')
ax2.tick_params(axis='y', labelcolor=color_rain)

# 4. Finishing
plt.title('Perbandingan Tren Curah Hujan Maksimal vs Frekuensi Klaim', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:

# 1. Menyiapkan data dan menghapus nilai kosong
df_stat = monthly_data_clean[['Max_Daily_Rainfall_Jkt', 'Claim_Frequency']].dropna()

# 2. Menghitung Korelasi Pearson dan Signifikansi
corr_p, p_val_p = pearsonr(df_stat['Max_Daily_Rainfall_Jkt'], df_stat['Claim_Frequency'])

# 3. Menghitung Korelasi Spearman dan Signifikansi
corr_s, p_val_s = spearmanr(df_stat['Max_Daily_Rainfall_Jkt'], df_stat['Claim_Frequency'])

# 4. Menampilkan Hasil
print(f"Korelasi Pearson: {corr_p:.4f} (p-value: {p_val_p:.4f})")
print(f"Korelasi Spearman: {corr_s:.4f} (p-value: {p_val_s:.4f})")

### Time Index

In [ ]:
# 1. Menyiapkan data (Gunakan Time_Index sebagai sumbu X)
x = monthly_data['Time_Index']
y = monthly_data['Total_Claim']

# 2. Plotting perbandingan tren
plt.figure(figsize=(12, 6))
sns.scatterplot(x=x, y=y, alpha=0.6, label='Data Observasi')

# --- Garis Tren Linier (Orde 1) ---
sns.regplot(x=x, y=y, order=1, scatter=False, color='red', 
            label='Tren Linier (Orde 1)', line_kws={'linestyle':'--'})

# --- Garis Tren Kuadratik (Orde 2) ---
sns.regplot(x=x, y=y, order=2, scatter=False, color='green', 
            label='Tren Kuadratik (Orde 2)', line_kws={'linewidth':3})


plt.xlabel('Indeks Waktu (Bulan)', fontsize=12)
plt.ylabel('Total Claim', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Feature Selection & Hyperparameter Tuning

## EC Feature Selection Catboost

In [ ]:


# baseline = {}
# for year, month in [(2025, 8), (2025, 9)]:
#     m_id = f"{year}_{month:02d}"
#     freq_val = fitness.loc[fitness['id'] == f"{m_id}_Claim_Frequency", 'value'].values[0]
#     sev_val = fitness.loc[fitness['id'] == f"{m_id}_Claim_Severity", 'value'].values[0]
#     baseline[(year, month)] = {'Freq': freq_val, 'Sev': sev_val}

# ALL_FEATURES = [
#     'Month_Num', 'month_sin', 'month_cos', 'Lag_1_Sev', 'Lag_1_Total', 'Lag_1_Freq', 
#     'Lag_2_Sev', 'Lag_2_Total', 'Lag_2_Freq', 'Lag_3_Sev', 'Lag_3_Total', 'Lag_3_Freq', 
#     'Lag_4_Sev', 'Lag_4_Total', 'Lag_4_Freq', 'Total_Rainfall_mm_Jkt', 'Avg_Daily_Rainfall_Jkt', 
#     'Max_Daily_Rainfall_Jkt', 'Rainy_Days_Count_Jkt', 'Extreme_Rain_Days_Jkt', 
#     'Total_Rainfall_mm_Sby', 'Avg_Daily_Rainfall_Sby', 'Max_Daily_Rainfall_Sby', 
#     'Rainy_Days_Count_Sby', 'Extreme_Rain_Days_Sby', 'RM_2_Freq', 'RM_2_Total', 'RM_2_Sev', 
#     'RM_3_Total', 'RM_3_Freq', 'RM_3_Sev', 'RM_4_Freq', 'RM_4_Total', 'RM_4_Sev', 
#     'RStd_2_Freq', 'RStd_2_Total', 'RStd_2_Sev', 'RStd_3_Total', 'RStd_3_Freq', 
#     'RStd_3_Sev', 'RStd_4_Freq', 'RStd_4_Total', 'RStd_4_Sev', 'Time_Index', 
#     'Time_Index_Squared', 'Quarter', 'Is_Year_End', 'Diff_1_Total', 'Diff_1_Freq', 
#     'Diff_1_Sev', 'Z_Score_Freq', 'Lag_1_Diff_1_Freq', 'Acc_Freq', 'Min_3_Freq', 
#     'Max_3_Freq', 'Stoch_K_Freq', 'MACD_Freq', 'Days_in_Month', 'Public_Holidays', 
#     'Working_Days', 'Momentum_2_Freq', 'Growth_Rate_Freq', 'Rain_x_LagFreq_Jkt', 
#     'Rain_x_LagFreq_Sby'
# ]

# best_hp = {
#     'Claim_Frequency': {'iterations': 200, 'depth': 3, 'learning_rate': 0.08, 'l2_leaf_reg': 5},
#     'Claim_Severity': {'iterations': 300, 'depth': 4, 'learning_rate': 0.08, 'l2_leaf_reg': 15}
# }

# BEST_BASELINE = {
#     'Claim_Frequency': ['Lag_4_Total', 'Working_Days', 'RM_2_Freq', 'Time_Index_Squared'],
#     'Claim_Severity': ['Max_Daily_Rainfall_Jkt', 'Lag_2_Freq', 'month_cos', 'Lag_2_Total', 'Lag_4_Freq']
# }

# # ==========================================
# # 2. HELPER FORECAST REKURSIF (ANTI-FREEZING)
# # ==========================================
# def run_recursive_forecast(models, df_base, forecast_tuples, features_freq, features_sev):
#     current_df = df_base.copy()
#     pred_results = {'Freq': [], 'Sev': [], 'Total': []}
    
#     for year, month in forecast_tuples:
#         last_row = current_df.iloc[-1].copy()
        
#         # Update Time dan Kalender
#         last_row['Month_Num'], last_row['Year_Num'] = month, year
#         last_row['month_sin'], last_row['month_cos'] = np.sin(2 * np.pi * month / 12), np.cos(2 * np.pi * month / 12)
#         last_row['Days_in_Month'] = calendar.monthrange(year, month)[1]
#         last_row['Public_Holidays'] = count_public_holidays(year, month)
#         last_row['Working_Days'] = count_working_days(year, month)
#         last_row['Quarter'] = (month - 1) // 3 + 1
#         last_row['Is_Year_End'] = 1 if month == 12 else 0
        
#         # Update Cuaca JKT dan SBY
#         for city, df_weather in [('Jkt', monthly_rainfall_jkt), ('Sby', monthly_rainfall_sby)]:
#             rf = df_weather[(df_weather['year'] == year) & (df_weather['month'] == month)]
#             cols = ['Total_Rainfall_mm_', 'Rainy_Days_Count_', 'Max_Daily_Rainfall_', 'Avg_Daily_Rainfall_', 'Extreme_Rain_Days_']
#             if not rf.empty:
#                 for c in cols: last_row[c + city] = rf[c + city].values[0]
#             else:
#                 for c in cols: last_row[c + city] = 0

#         # Update Time Index
#         last_row['Time_Index'] = current_df.iloc[-1]['Time_Index'] + 1
#         last_row['Time_Index_Squared'] = last_row['Time_Index'] ** 2
        
#         # Update Lags (Anti-Leakage: ambil dari current_df)
#         for lag in range(1, 5):
#             last_row[f'Lag_{lag}_Total'] = current_df.iloc[-lag]['Total_Claim']
#             last_row[f'Lag_{lag}_Freq'] = current_df.iloc[-lag]['Claim_Frequency']
#             last_row[f'Lag_{lag}_Sev'] = current_df.iloc[-lag]['Claim_Severity']

#         # Update Diffs dan Technical Indicators (Anti-Freezing)
#         last_row['Diff_1_Total'] = current_df.iloc[-1]['Total_Claim'] - current_df.iloc[-2]['Total_Claim'] 
#         last_row['Diff_1_Sev'] = current_df.iloc[-1]['Claim_Severity'] - current_df.iloc[-2]['Claim_Severity']
#         last_row['Diff_1_Freq'] = current_df.iloc[-1]['Claim_Frequency'] - current_df.iloc[-2]['Claim_Frequency']
#         last_row['Lag_1_Diff_1_Freq'] = current_df.iloc[-2]['Claim_Frequency'] - current_df.iloc[-3]['Claim_Frequency']
#         last_row['Acc_Freq'] = last_row['Diff_1_Freq'] - last_row['Lag_1_Diff_1_Freq']
#         last_row['Growth_Rate_Freq'] = (last_row['Diff_1_Freq'] / (last_row['Lag_2_Freq'] + 1e-6)) * 100
#         last_row['Momentum_2_Freq'] = last_row['Lag_1_Freq'] - last_row['Lag_3_Freq']

#         for rm in [2, 3, 4]:
#             for t, col_src in [('Freq', 'Claim_Frequency'), ('Sev', 'Claim_Severity'), ('Total', 'Total_Claim')]:
#                 last_row[f'RM_{rm}_{t}'] = current_df.iloc[-rm:][col_src].mean()
#                 last_row[f'RStd_{rm}_{t}'] = current_df.iloc[-rm:][col_src].std()

#         last_row['Min_3_Freq'] = current_df.iloc[-3:]['Claim_Frequency'].min()
#         last_row['Max_3_Freq'] = current_df.iloc[-3:]['Claim_Frequency'].max()
#         last_row['MACD_Freq'] = last_row['RM_2_Freq'] - last_row['RM_4_Freq']
#         last_row['Z_Score_Freq'] = (last_row['Lag_1_Freq'] - last_row['RM_4_Freq']) / (last_row['RStd_4_Freq'] + 1e-6)
#         last_row['Stoch_K_Freq'] = (last_row['Lag_1_Freq'] - last_row['Min_3_Freq']) / (last_row['Max_3_Freq'] - last_row['Min_3_Freq'] + 1e-6)
#         last_row['Rain_x_LagFreq_Jkt'] = last_row['Total_Rainfall_mm_Jkt'] * last_row['Lag_1_Freq']
#         last_row['Rain_x_LagFreq_Sby'] = last_row['Total_Rainfall_mm_Sby'] * last_row['Lag_1_Freq']

#         # Prediksi Step Saat Ini
#         X_f = pd.DataFrame([last_row[features_freq]])
#         pf = max(1, models['Claim_Frequency'].predict(X_f)[0])
#         X_s = pd.DataFrame([last_row[features_sev]])
#         ps = max(1e-6, models['Claim_Severity'].predict(X_s)[0])
#         pt = pf * ps
        
#         pred_results['Freq'].append(pf); pred_results['Sev'].append(ps); pred_results['Total'].append(pt)
        
#         # Masukkan hasil ke current_df untuk step berikutnya
#         new_entry = last_row.copy()
#         new_entry['Claim_Frequency'], new_entry['Claim_Severity'], new_entry['Total_Claim'] = pf, ps, pt
#         current_df = pd.concat([current_df, pd.DataFrame([new_entry])], ignore_index=True)
        
#     return pred_results

# # ==========================================
# # 3. EVALUASI FITNESS (PRIORITAS KAGGLE)
# # ==========================================
# def evaluate_fitness_robust(features_freq, features_sev, full_df):
#     if len(features_freq) == 0 or len(features_sev) == 0:
#         return 999.0, 999.0, 999.0
    
#     try:
#         def get_mape(actual, pred): return np.mean(np.abs((np.array(actual) - np.array(pred)) / np.array(actual))) * 100
        
#         # 1. Validasi Lokal (5 Bulan Terakhir)
#         local_train, local_valid = full_df.iloc[:-5].copy(), full_df.iloc[-5:].copy()
#         models_l = {}
#         for t in ['Claim_Frequency', 'Claim_Severity']:
#             m = CatBoostRegressor(**best_hp[t], loss_function='MAPE', random_seed=42, verbose=0)
#             m.fit(local_train[features_freq if 'Freq' in t else features_sev], local_train[t])
#             models_l[t] = m
        
#         l_months = [(r['Year_Num'], r['Month_Num']) for _, r in local_valid.iterrows()]
#         res_l = run_recursive_forecast(models_l, local_train, l_months, features_freq, features_sev)
#         mape_l = (get_mape(local_valid['Claim_Frequency'], res_l['Freq']) + get_mape(local_valid['Claim_Severity'], res_l['Sev']) + get_mape(local_valid['Total_Claim'], res_l['Total'])) / 3

#         # 2. Validasi Kaggle LB (2 Bulan)
#         models_k = {}
#         for t in ['Claim_Frequency', 'Claim_Severity']:
#             m = CatBoostRegressor(**best_hp[t], loss_function='MAPE', random_seed=42, verbose=0)
#             m.fit(full_df[features_freq if 'Freq' in t else features_sev], full_df[t])
#             models_k[t] = m
            
#         res_k = run_recursive_forecast(models_k, full_df, [(2025, 8), (2025, 9)], features_freq, features_sev)
#         base_f = [baseline[(2025,8)]['Freq'], baseline[(2025,9)]['Freq']]
#         base_s = [baseline[(2025,8)]['Sev'], baseline[(2025,9)]['Sev']]
#         mape_k = (get_mape(base_f, res_k['Freq']) + get_mape(base_s, res_k['Sev']) + get_mape([f*s for f,s in zip(base_f, base_s)], res_k['Total'])) / 3

#         # --- LOGIKA SELEKSI CUSTOM ---
#         gap = abs(mape_l - mape_k)
#         if gap <= 2.0:
#             fitness = mape_k  # Prioritas skor Kaggle
#         else:
#             fitness = mape_k + (gap * 5) # Penalti berat jika tidak robust
            
#         return fitness, mape_l, mape_k

#     except:
#         return 999.0, 999.0, 999.0

# # ==========================================
# # 4. ENGINE EVOLUSI
# # ==========================================
# def run_evolution(df_train, pop_size=50, generations=10):
#     print(f"Memulai Evolusi Feature Selection: {pop_size} Pop, {generations} Gen")
    
#     population = [BEST_BASELINE]
#     for _ in range(pop_size - 1):
#         population.append({
#             'Claim_Frequency': random.sample(ALL_FEATURES, random.randint(10, 20)),
#             'Claim_Severity': random.sample(ALL_FEATURES, random.randint(10, 20))
#         })

#     best_dna, best_lb_overall = None, 999.0

#     for gen in range(generations):
#         scored = []
#         for ind in population:
#             fit, loc, lb = evaluate_fitness_robust(ind['Claim_Frequency'], ind['Claim_Severity'], df_train)
#             scored.append({'fitness': fit, 'local': loc, 'lb': lb, 'dna': ind})
        
#         scored.sort(key=lambda x: x['fitness'])
#         current_best = scored[0]
        
#         if current_best['lb'] < best_lb_overall:
#             best_lb_overall, best_dna = current_best['lb'], current_best['dna']
        
#         print(f"MAPE: {current_best['local']:.4f} ")

#         next_gen = [scored[0]['dna'], scored[1]['dna']]
#         while len(next_gen) < pop_size:
#             p1, p2 = random.choice(scored[:5])['dna'], random.choice(scored[:5])['dna']
#             child = {}
#             for t in ['Claim_Frequency', 'Claim_Severity']:
#                 c_feats = list(set([f for f in (p1[t] + p2[t]) if random.random() < 0.5]))
#                 if random.random() < 0.1: # Mutasi
#                     if random.random() < 0.5 and len(c_feats) > 5: c_feats.remove(random.choice(c_feats))
#                     else: c_feats.append(random.choice(ALL_FEATURES))
#                 child[t] = list(set(c_feats))
#             next_gen.append(child)
#         population = next_gen

#     print("\nPROSES EVOLUSI SELESAI")
#     print(f"Fitur Terbaik Claim_Frequency:\n{best_dna['Claim_Frequency']}")
#     print(f"\nFitur Terbaik Claim_Severity:\n{best_dna['Claim_Severity']}")
#     return best_dna

# monthly_data_clean = monthly_data_clean.sort_values("YearMonth").reset_index(drop=True)
# best_features = run_evolution(monthly_data_clean, pop_size=2, generations=5)

## EC Feature Selection LGBM

In [ ]:
# for year, month in [(2025, 8), (2025, 9)]:
#     m_id = f"{year}_{month:02d}"
#     freq_val = fitness.loc[fitness['id'] == f"{m_id}_Claim_Frequency", 'value'].values[0]
#     sev_val = fitness.loc[fitness['id'] == f"{m_id}_Claim_Severity", 'value'].values[0]
#     baseline[(year, month)] = {'Freq': freq_val, 'Sev': sev_val}

# ALL_FEATURES = [
#     'Month_Num', 'month_sin', 'month_cos', 'Lag_1_Sev', 'Lag_1_Total', 'Lag_1_Freq', 
#     'Lag_2_Sev', 'Lag_2_Total', 'Lag_2_Freq', 'Lag_3_Sev', 'Lag_3_Total', 'Lag_3_Freq', 
#     'Lag_4_Sev', 'Lag_4_Total', 'Lag_4_Freq', 'Total_Rainfall_mm_Jkt', 'Avg_Daily_Rainfall_Jkt', 
#     'Max_Daily_Rainfall_Jkt', 'Rainy_Days_Count_Jkt', 'Extreme_Rain_Days_Jkt', 
#     'Total_Rainfall_mm_Sby', 'Avg_Daily_Rainfall_Sby', 'Max_Daily_Rainfall_Sby', 
#     'Rainy_Days_Count_Sby', 'Extreme_Rain_Days_Sby', 'RM_2_Freq', 'RM_2_Total', 'RM_2_Sev', 
#     'RM_3_Total', 'RM_3_Freq', 'RM_3_Sev', 'RM_4_Freq', 'RM_4_Total', 'RM_4_Sev', 
#     'RStd_2_Freq', 'RStd_2_Total', 'RStd_2_Sev', 'RStd_3_Total', 'RStd_3_Freq', 
#     'RStd_3_Sev', 'RStd_4_Freq', 'RStd_4_Total', 'RStd_4_Sev', 'Time_Index', 
#     'Time_Index_Squared', 'Quarter', 'Is_Year_End', 'Diff_1_Total', 'Diff_1_Freq', 
#     'Diff_1_Sev', 'Z_Score_Freq', 'Lag_1_Diff_1_Freq', 'Acc_Freq', 'Min_3_Freq', 
#     'Max_3_Freq', 'Stoch_K_Freq', 'MACD_Freq', 'Days_in_Month', 'Public_Holidays', 
#     'Working_Days', 'Momentum_2_Freq', 'Growth_Rate_Freq', 'Rain_x_LagFreq_Jkt', 
#     'Rain_x_LagFreq_Sby'
# ]

# best_hp = {
#     'Claim_Frequency': {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.03, 'reg_lambda': 3},
#     'Claim_Severity': {'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.03, 'reg_lambda': 5}
# }

# BEST_BASELINE = {
#      'Claim_Frequency': ['Lag_4_Total', 'Working_Days', 'RM_2_Freq', 'Time_Index_Squared'],
#     'Claim_Severity': ['Max_Daily_Rainfall_Jkt', 'Lag_2_Freq', 'month_cos', 'Lag_2_Total', 'Lag_4_Freq']
# }

# # ==========================================
# # 2. HELPER FORECAST REKURSIF (ANTI-FREEZING)
# # ==========================================
# def run_recursive_forecast(models, df_base, forecast_tuples, features_freq, features_sev):
#     current_df = df_base.copy()
#     pred_results = {'Freq': [], 'Sev': [], 'Total': []}
    
#     for year, month in forecast_tuples:
#         last_row = current_df.iloc[-1].copy()
        
#         # Update Time dan Kalender
#         last_row['Month_Num'], last_row['Year_Num'] = month, year
#         last_row['month_sin'], last_row['month_cos'] = np.sin(2 * np.pi * month / 12), np.cos(2 * np.pi * month / 12)
#         last_row['Days_in_Month'] = calendar.monthrange(year, month)[1]
#         last_row['Public_Holidays'] = count_public_holidays(year, month)
#         last_row['Working_Days'] = count_working_days(year, month)
#         last_row['Quarter'] = (month - 1) // 3 + 1
#         last_row['Is_Year_End'] = 1 if month == 12 else 0
        
#         # Update Cuaca JKT dan SBY
#         for city, df_weather in [('Jkt', monthly_rainfall_jkt), ('Sby', monthly_rainfall_sby)]:
#             rf = df_weather[(df_weather['year'] == year) & (df_weather['month'] == month)]
#             cols = ['Total_Rainfall_mm_', 'Rainy_Days_Count_', 'Max_Daily_Rainfall_', 'Avg_Daily_Rainfall_', 'Extreme_Rain_Days_']
#             if not rf.empty:
#                 for c in cols: last_row[c + city] = rf[c + city].values[0]
#             else:
#                 for c in cols: last_row[c + city] = 0

#         # Update Time Index
#         last_row['Time_Index'] = current_df.iloc[-1]['Time_Index'] + 1
#         last_row['Time_Index_Squared'] = last_row['Time_Index'] ** 2
        
#         # Update Lags (Anti-Leakage: ambil dari current_df)
#         for lag in range(1, 5):
#             last_row[f'Lag_{lag}_Total'] = current_df.iloc[-lag]['Total_Claim']
#             last_row[f'Lag_{lag}_Freq'] = current_df.iloc[-lag]['Claim_Frequency']
#             last_row[f'Lag_{lag}_Sev'] = current_df.iloc[-lag]['Claim_Severity']

#         # Update Diffs dan Technical Indicators (Anti-Freezing)
#         last_row['Diff_1_Total'] = current_df.iloc[-1]['Total_Claim'] - current_df.iloc[-2]['Total_Claim'] 
#         last_row['Diff_1_Sev'] = current_df.iloc[-1]['Claim_Severity'] - current_df.iloc[-2]['Claim_Severity']
#         last_row['Diff_1_Freq'] = current_df.iloc[-1]['Claim_Frequency'] - current_df.iloc[-2]['Claim_Frequency']
#         last_row['Lag_1_Diff_1_Freq'] = current_df.iloc[-2]['Claim_Frequency'] - current_df.iloc[-3]['Claim_Frequency']
#         last_row['Acc_Freq'] = last_row['Diff_1_Freq'] - last_row['Lag_1_Diff_1_Freq']
#         last_row['Growth_Rate_Freq'] = (last_row['Diff_1_Freq'] / (last_row['Lag_2_Freq'] + 1e-6)) * 100
#         last_row['Momentum_2_Freq'] = last_row['Lag_1_Freq'] - last_row['Lag_3_Freq']

#         for rm in [2, 3, 4]:
#             for t, col_src in [('Freq', 'Claim_Frequency'), ('Sev', 'Claim_Severity'), ('Total', 'Total_Claim')]:
#                 last_row[f'RM_{rm}_{t}'] = current_df.iloc[-rm:][col_src].mean()
#                 last_row[f'RStd_{rm}_{t}'] = current_df.iloc[-rm:][col_src].std()

#         last_row['Min_3_Freq'] = current_df.iloc[-3:]['Claim_Frequency'].min()
#         last_row['Max_3_Freq'] = current_df.iloc[-3:]['Claim_Frequency'].max()
#         last_row['MACD_Freq'] = last_row['RM_2_Freq'] - last_row['RM_4_Freq']
#         last_row['Z_Score_Freq'] = (last_row['Lag_1_Freq'] - last_row['RM_4_Freq']) / (last_row['RStd_4_Freq'] + 1e-6)
#         last_row['Stoch_K_Freq'] = (last_row['Lag_1_Freq'] - last_row['Min_3_Freq']) / (last_row['Max_3_Freq'] - last_row['Min_3_Freq'] + 1e-6)
#         last_row['Rain_x_LagFreq_Jkt'] = last_row['Total_Rainfall_mm_Jkt'] * last_row['Lag_1_Freq']
#         last_row['Rain_x_LagFreq_Sby'] = last_row['Total_Rainfall_mm_Sby'] * last_row['Lag_1_Freq']

#         # Prediksi Step Saat Ini
#         X_f = pd.DataFrame([last_row[features_freq]])
#         pf = max(1, models['Claim_Frequency'].predict(X_f)[0])
#         X_s = pd.DataFrame([last_row[features_sev]])
#         ps = max(1e-6, models['Claim_Severity'].predict(X_s)[0])
#         pt = pf * ps
        
#         pred_results['Freq'].append(pf); pred_results['Sev'].append(ps); pred_results['Total'].append(pt)
        
#         # Masukkan hasil ke current_df untuk step berikutnya
#         new_entry = last_row.copy()
#         new_entry['Claim_Frequency'], new_entry['Claim_Severity'], new_entry['Total_Claim'] = pf, ps, pt
#         current_df = pd.concat([current_df, pd.DataFrame([new_entry])], ignore_index=True)
        
#     return pred_results

# # ==========================================
# # 3. EVALUASI FITNESS (PRIORITAS KAGGLE)
# # ==========================================
# def evaluate_fitness_robust(features_freq, features_sev, full_df):
#     if len(features_freq) == 0 or len(features_sev) == 0:
#         return 999.0, 999.0, 999.0
    
#     try:
#         def get_mape(actual, pred): 
#             return np.mean(np.abs((np.array(actual) - np.array(pred)) / (np.array(actual) + 1e-6))) * 100
        
#         # Validasi Lokal
#         local_train, local_valid = full_df.iloc[:-5].copy(), full_df.iloc[-5:].copy()
#         models_l = {}
#         for t in ['Claim_Frequency', 'Claim_Severity']:
#             m = LGBMRegressor(**best_hp[t], objective='mape', random_state=42, verbosity=-1)
#             m.fit(local_train[features_freq if 'Freq' in t else features_sev], local_train[t])
#             models_l[t] = m
        
#         l_months = [(r['Year_Num'], r['Month_Num']) for _, r in local_valid.iterrows()]
#         res_l = run_recursive_forecast(models_l, local_train, l_months, features_freq, features_sev)
#         mape_l = (get_mape(local_valid['Claim_Frequency'], res_l['Freq']) + 
#                   get_mape(local_valid['Claim_Severity'], res_l['Sev'])) / 2

#         # Validasi Leadboard
#         models_k = {}
#         for t in ['Claim_Frequency', 'Claim_Severity']:
#             m = LGBMRegressor(**best_hp[t], objective='mape', random_state=42, verbosity=-1)
#             m.fit(full_df[features_freq if 'Freq' in t else features_sev], full_df[t])
#             models_k[t] = m
            
#         res_k = run_recursive_forecast(models_k, full_df, [(2025, 8), (2025, 9)], features_freq, features_sev)
#         base_f = [baseline[(2025,8)]['Freq'], baseline[(2025,9)]['Freq']]
#         base_s = [baseline[(2025,8)]['Sev'], baseline[(2025,9)]['Sev']]
#         mape_k = (get_mape(base_f, res_k['Freq']) + get_mape(base_s, res_k['Sev'])) / 2

#         gap = abs(mape_l - mape_k)
#         fitness = mape_k if gap <= 2.0 else mape_k + (gap * 5)
            
#         return fitness, mape_l, mape_k

#     except Exception:
#         return 999.0, 999.0, 999.0

# # ==========================================
# # 4. ENGINE EVOLUSI
# # ==========================================
# def run_evolution(df_train, pop_size=50, generations=10):
#     print(f"Memulai Evolusi Feature Selection: {pop_size} Pop, {generations} Gen")
    
#     population = [BEST_BASELINE]
#     for _ in range(pop_size - 1):
#         population.append({
#             'Claim_Frequency': random.sample(ALL_FEATURES, random.randint(10, 20)),
#             'Claim_Severity': random.sample(ALL_FEATURES, random.randint(10, 20))
#         })

#     best_dna, best_lb_overall = None, 999.0

#     for gen in range(generations):
#         scored = []
#         for ind in population:
#             fit, loc, lb = evaluate_fitness_robust(ind['Claim_Frequency'], ind['Claim_Severity'], df_train)
#             scored.append({'fitness': fit, 'local': loc, 'lb': lb, 'dna': ind})
        
#         scored.sort(key=lambda x: x['fitness'])
#         current_best = scored[0]
        
#         if current_best['lb'] < best_lb_overall:
#             best_lb_overall, best_dna = current_best['lb'], current_best['dna']
        
#         print(f"MAPE: {current_best['local']:.4f}")

#         next_gen = [scored[0]['dna'], scored[1]['dna']]
#         while len(next_gen) < pop_size:
#             p1, p2 = random.choice(scored[:5])['dna'], random.choice(scored[:5])['dna']
#             child = {}
#             for t in ['Claim_Frequency', 'Claim_Severity']:
#                 c_feats = list(set([f for f in (p1[t] + p2[t]) if random.random() < 0.5]))
#                 if random.random() < 0.1: # Mutasi
#                     if random.random() < 0.5 and len(c_feats) > 5: c_feats.remove(random.choice(c_feats))
#                     else: c_feats.append(random.choice(ALL_FEATURES))
#                 child[t] = list(set(c_feats))
#             next_gen.append(child)
#         population = next_gen

#     print("\nPROSES EVOLUSI SELESAI")
#     print(f"Fitur Terbaik Claim_Frequency:\n{best_dna['Claim_Frequency']}")
#     print(f"\nFitur Terbaik Claim_Severity:\n{best_dna['Claim_Severity']}")
#     return best_dna


# monthly_data_clean = monthly_data_clean.sort_values("YearMonth").reset_index(drop=True)
# best_features = run_evolution(monthly_data_clean, pop_size=2, generations=5)

## EC Feature Selection XGBoost

In [ ]:
# for year, month in [(2025, 8), (2025, 9)]:
#     m_id = f"{year}_{month:02d}"
#     freq_val = fitness.loc[fitness['id'] == f"{m_id}_Claim_Frequency", 'value'].values[0]
#     sev_val = fitness.loc[fitness['id'] == f"{m_id}_Claim_Severity", 'value'].values[0]
#     baseline[(year, month)] = {'Freq': freq_val, 'Sev': sev_val}

# ALL_FEATURES = [
#     'Month_Num', 'month_sin', 'month_cos', 'Lag_1_Sev', 'Lag_1_Total', 'Lag_1_Freq', 
#     'Lag_2_Sev', 'Lag_2_Total', 'Lag_2_Freq', 'Lag_3_Sev', 'Lag_3_Total', 'Lag_3_Freq', 
#     'Lag_4_Sev', 'Lag_4_Total', 'Lag_4_Freq', 'Total_Rainfall_mm_Jkt', 'Avg_Daily_Rainfall_Jkt', 
#     'Max_Daily_Rainfall_Jkt', 'Rainy_Days_Count_Jkt', 'Extreme_Rain_Days_Jkt', 
#     'Total_Rainfall_mm_Sby', 'Avg_Daily_Rainfall_Sby', 'Max_Daily_Rainfall_Sby', 
#     'Rainy_Days_Count_Sby', 'Extreme_Rain_Days_Sby', 'RM_2_Freq', 'RM_2_Total', 'RM_2_Sev', 
#     'RM_3_Total', 'RM_3_Freq', 'RM_3_Sev', 'RM_4_Freq', 'RM_4_Total', 'RM_4_Sev', 
#     'RStd_2_Freq', 'RStd_2_Total', 'RStd_2_Sev', 'RStd_3_Total', 'RStd_3_Freq', 
#     'RStd_3_Sev', 'RStd_4_Freq', 'RStd_4_Total', 'RStd_4_Sev', 'Time_Index', 
#     'Time_Index_Squared', 'Quarter', 'Is_Year_End', 'Diff_1_Total', 'Diff_1_Freq', 
#     'Diff_1_Sev', 'Z_Score_Freq', 'Lag_1_Diff_1_Freq', 'Acc_Freq', 'Min_3_Freq', 
#     'Max_3_Freq', 'Stoch_K_Freq', 'MACD_Freq', 'Days_in_Month', 'Public_Holidays', 
#     'Working_Days', 'Momentum_2_Freq', 'Growth_Rate_Freq', 'Rain_x_LagFreq_Jkt', 
#     'Rain_x_LagFreq_Sby'
# ]

# best_hp = {
#     'Claim_Frequency': {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.03, 'reg_lambda': 3},
#     'Claim_Severity': {'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.03, 'reg_lambda': 5}
# }

# BEST_BASELINE = {
#      'Claim_Frequency': ['Lag_4_Total', 'Working_Days', 'RM_2_Freq', 'Time_Index_Squared'],
#     'Claim_Severity': ['Max_Daily_Rainfall_Jkt', 'Lag_2_Freq', 'month_cos', 'Lag_2_Total', 'Lag_4_Freq']
# }

# # ==========================================
# # 2. HELPER FORECAST REKURSIF (ANTI-FREEZING)
# # ==========================================
# def run_recursive_forecast(models, df_base, forecast_tuples, features_freq, features_sev):
#     current_df = df_base.copy()
#     pred_results = {'Freq': [], 'Sev': [], 'Total': []}
    
#     for year, month in forecast_tuples:
#         last_row = current_df.iloc[-1].copy()
        
#         # Update Time dan Kalender
#         last_row['Month_Num'], last_row['Year_Num'] = month, year
#         last_row['month_sin'], last_row['month_cos'] = np.sin(2 * np.pi * month / 12), np.cos(2 * np.pi * month / 12)
#         last_row['Days_in_Month'] = calendar.monthrange(year, month)[1]
#         last_row['Public_Holidays'] = count_public_holidays(year, month)
#         last_row['Working_Days'] = count_working_days(year, month)
#         last_row['Quarter'] = (month - 1) // 3 + 1
#         last_row['Is_Year_End'] = 1 if month == 12 else 0
        
#         # Update Cuaca JKT dan SBY
#         for city, df_weather in [('Jkt', monthly_rainfall_jkt), ('Sby', monthly_rainfall_sby)]:
#             rf = df_weather[(df_weather['year'] == year) & (df_weather['month'] == month)]
#             cols = ['Total_Rainfall_mm_', 'Rainy_Days_Count_', 'Max_Daily_Rainfall_', 'Avg_Daily_Rainfall_', 'Extreme_Rain_Days_']
#             if not rf.empty:
#                 for c in cols: last_row[c + city] = rf[c + city].values[0]
#             else:
#                 for c in cols: last_row[c + city] = 0

#         # Update Time Index
#         last_row['Time_Index'] = current_df.iloc[-1]['Time_Index'] + 1
#         last_row['Time_Index_Squared'] = last_row['Time_Index'] ** 2
        
#         # Update Lags (Anti-Leakage: ambil dari current_df)
#         for lag in range(1, 5):
#             last_row[f'Lag_{lag}_Total'] = current_df.iloc[-lag]['Total_Claim']
#             last_row[f'Lag_{lag}_Freq'] = current_df.iloc[-lag]['Claim_Frequency']
#             last_row[f'Lag_{lag}_Sev'] = current_df.iloc[-lag]['Claim_Severity']

#         # Update Diffs dan Technical Indicators (Anti-Freezing)
#         last_row['Diff_1_Total'] = current_df.iloc[-1]['Total_Claim'] - current_df.iloc[-2]['Total_Claim'] 
#         last_row['Diff_1_Sev'] = current_df.iloc[-1]['Claim_Severity'] - current_df.iloc[-2]['Claim_Severity']
#         last_row['Diff_1_Freq'] = current_df.iloc[-1]['Claim_Frequency'] - current_df.iloc[-2]['Claim_Frequency']
#         last_row['Lag_1_Diff_1_Freq'] = current_df.iloc[-2]['Claim_Frequency'] - current_df.iloc[-3]['Claim_Frequency']
#         last_row['Acc_Freq'] = last_row['Diff_1_Freq'] - last_row['Lag_1_Diff_1_Freq']
#         last_row['Growth_Rate_Freq'] = (last_row['Diff_1_Freq'] / (last_row['Lag_2_Freq'] + 1e-6)) * 100
#         last_row['Momentum_2_Freq'] = last_row['Lag_1_Freq'] - last_row['Lag_3_Freq']

#         for rm in [2, 3, 4]:
#             for t, col_src in [('Freq', 'Claim_Frequency'), ('Sev', 'Claim_Severity'), ('Total', 'Total_Claim')]:
#                 last_row[f'RM_{rm}_{t}'] = current_df.iloc[-rm:][col_src].mean()
#                 last_row[f'RStd_{rm}_{t}'] = current_df.iloc[-rm:][col_src].std()

#         last_row['Min_3_Freq'] = current_df.iloc[-3:]['Claim_Frequency'].min()
#         last_row['Max_3_Freq'] = current_df.iloc[-3:]['Claim_Frequency'].max()
#         last_row['MACD_Freq'] = last_row['RM_2_Freq'] - last_row['RM_4_Freq']
#         last_row['Z_Score_Freq'] = (last_row['Lag_1_Freq'] - last_row['RM_4_Freq']) / (last_row['RStd_4_Freq'] + 1e-6)
#         last_row['Stoch_K_Freq'] = (last_row['Lag_1_Freq'] - last_row['Min_3_Freq']) / (last_row['Max_3_Freq'] - last_row['Min_3_Freq'] + 1e-6)
#         last_row['Rain_x_LagFreq_Jkt'] = last_row['Total_Rainfall_mm_Jkt'] * last_row['Lag_1_Freq']
#         last_row['Rain_x_LagFreq_Sby'] = last_row['Total_Rainfall_mm_Sby'] * last_row['Lag_1_Freq']

#         # Prediksi Step Saat Ini
#         X_f = pd.DataFrame([last_row[features_freq]])
#         pf = max(1, models['Claim_Frequency'].predict(X_f)[0])
#         X_s = pd.DataFrame([last_row[features_sev]])
#         ps = max(1e-6, models['Claim_Severity'].predict(X_s)[0])
#         pt = pf * ps
        
#         pred_results['Freq'].append(pf); pred_results['Sev'].append(ps); pred_results['Total'].append(pt)
        
#         # Masukkan hasil ke current_df untuk step berikutnya
#         new_entry = last_row.copy()
#         new_entry['Claim_Frequency'], new_entry['Claim_Severity'], new_entry['Total_Claim'] = pf, ps, pt
#         current_df = pd.concat([current_df, pd.DataFrame([new_entry])], ignore_index=True)
        
#     return pred_results

# # ==========================================
# # 3. EVALUASI FITNESS (PRIORITAS KAGGLE)
# def evaluate_fitness_robust(features_freq, features_sev, full_df):
#     if len(features_freq) == 0 or len(features_sev) == 0:
#         return 999.0, 999.0, 999.0
    
#     try:
#         def get_mape(actual, pred): 
#             return np.mean(np.abs((np.array(actual) - np.array(pred)) / (np.array(actual) + 1e-6))) * 100
        
#         # Validasi Lokal (Recursive)
#         local_train, local_valid = full_df.iloc[:-5].copy(), full_df.iloc[-5:].copy()
#         models_l = {}
#         for t in ['Claim_Frequency', 'Claim_Severity']:
#             m = XGBRegressor(**best_hp[t], objective='reg:absoluteerror', random_state=42, verbosity=0)
#             m.fit(local_train[features_freq if 'Freq' in t else features_sev], local_train[t])
#             models_l[t] = m
        
#         l_months = [(r['Year_Num'], r['Month_Num']) for _, r in local_valid.iterrows()]
#         res_l = run_recursive_forecast(models_l, local_train, l_months, features_freq, features_sev)
#         mape_l = (get_mape(local_valid['Claim_Frequency'], res_l['Freq']) + 
#                   get_mape(local_valid['Claim_Severity'], res_l['Sev'])) / 2

#         # Validasi Leadboard (Recursive)
#         models_k = {}
#         for t in ['Claim_Frequency', 'Claim_Severity']:
#             m = XGBRegressor(**best_hp[t], objective='reg:absoluteerror', random_state=42, verbosity=0)
#             m.fit(full_df[features_freq if 'Freq' in t else features_sev], full_df[t])
#             models_k[t] = m
            
#         res_k = run_recursive_forecast(models_k, full_df, [(2025, 8), (2025, 9)], features_freq, features_sev)
#         base_f = [baseline[(2025, 8)]['Freq'], baseline[(2025, 9)]['Freq']]
#         base_s = [baseline[(2025, 8)]['Sev'], baseline[(2025, 9)]['Sev']]
#         mape_k = (get_mape(base_f, res_k['Freq']) + get_mape(base_s, res_k['Sev'])) / 2

#         # Mekanisme Penalti Stabilitas
#         gap = abs(mape_l - mape_k)
#         fitness = mape_k if gap <= 2.0 else mape_k + (gap * 5)
            
#         return fitness, mape_l, mape_k
#     except Exception:
#         return 999.0, 999.0, 999.0

# # ==========================================
# # 4. ENGINE EVOLUSI
# # ==========================================
# def run_evolution(df_train, pop_size=50, generations=10):
#     print(f"Memulai Evolusi Feature Selection XGBoost: {pop_size} Pop, {generations} Gen")
    
#     # Inisialisasi populasi sepenuhnya acak (Tanpa Baseline)
#     population = []
#     for _ in range(pop_size):
#         population.append({
#             'Claim_Frequency': random.sample(ALL_FEATURES, random.randint(5, 15)),
#             'Claim_Severity': random.sample(ALL_FEATURES, random.randint(5, 15))
#         })

#     best_dna, best_lb_overall = None, 999.0

#     for gen in range(generations):
#         scored = []
#         for ind in population:
#             fit, loc, lb = evaluate_fitness_robust(ind['Claim_Frequency'], ind['Claim_Severity'], df_train)
#             scored.append({'fitness': fit, 'local': loc, 'lb': lb, 'dna': ind})
        
#         scored.sort(key=lambda x: x['fitness'])
#         if scored[0]['lb'] < best_lb_overall:
#             best_lb_overall, best_dna = scored[0]['lb'], scored[0]['dna']
        
#         print(f"MAPE: {scored[0]['local']:.4f}")

#         # Reproduksi (Elitisme + Crossover + Mutasi)
#         next_gen = [scored[0]['dna'], scored[1]['dna']]
#         while len(next_gen) < pop_size:
#             p1, p2 = random.choice(scored[:10])['dna'], random.choice(scored[:10])['dna']
#             child = {}
#             for t in ['Claim_Frequency', 'Claim_Severity']:
#                 c_feats = list(set([f for f in (p1[t] + p2[t]) if random.random() < 0.5]))
#                 if random.random() < 0.2: # Mutasi
#                     if random.random() < 0.5 and len(c_feats) > 3: 
#                         c_feats.remove(random.choice(c_feats))
#                     else: 
#                         c_feats.append(random.choice(ALL_FEATURES))
#                 child[t] = list(set(c_feats))
#             next_gen.append(child)
#         population = next_gen
        
#     print("\nPROSES EVOLUSI SELESAI")
#     print(f"Fitur Terbaik Claim_Frequency:\n{best_dna['Claim_Frequency']}")
#     print(f"\nFitur Terbaik Claim_Severity:\n{best_dna['Claim_Severity']}")
#     return best_dna


# monthly_data_clean = monthly_data_clean.sort_values("YearMonth").reset_index(drop=True)
# best_features = run_evolution(monthly_data_clean, pop_size=2, generations=5)

## EC Hyperparameter Tuning Catboost

In [ ]:

# feature_mapping = {
#     'Claim_Frequency': ['Lag_4_Total', 'Working_Days', 'RM_2_Freq', 'Time_Index_Squared'],
#     'Claim_Severity': ['Max_Daily_Rainfall_Jkt', 'Lag_2_Freq', 'month_cos', 'Lag_2_Total', 'Lag_4_Freq']
# }

# EXOG_COLS = ['Month_Num', 'Year_Num', 'month_sin', 'month_cos', 'Days_in_Month', 
#              'Working_Days', 'Quarter', 'Is_Year_End', 'Avg_Usia', 'Public_Holidays',
#              'Total_Rainfall_mm_Jkt', 'Max_Daily_Rainfall_Jkt', 'Avg_Daily_Rainfall_Jkt', 
#              'Extreme_Rain_Days_Jkt', 'Rainy_Days_Count_Jkt', 'Total_Rainfall_mm_Sby', 
#              'Max_Daily_Rainfall_Sby', 'Avg_Daily_Rainfall_Sby', 'Extreme_Rain_Days_Sby', 'Rainy_Days_Count_Sby']

# # BEST BASELINE DIUPDATE KE CATBOOST
# BEST_BASELINE_HP = {
#     'Claim_Frequency': {'iterations': 200, 'depth': 3, 'learning_rate': 0.08, 'l2_leaf_reg': 5},
#     'Claim_Severity':  {'iterations': 300, 'depth': 4, 'learning_rate': 0.08, 'l2_leaf_reg': 15}
# }

# # NAMA PARAMETER DIUBAH MENGIKUTI CATBOOST
# HP_SPACE = {
#     'iterations': [50, 100, 150, 200, 250, 300],
#     'depth': [3, 4, 5, 6, 7],
#     'learning_rate': [0.01, 0.03, 0.05, 0.08, 0.1],
#     'l2_leaf_reg': [1, 3, 5, 7, 10, 15]
# }

# # ==========================================
# # 2. HELPER & PREDIKSI REKURSIF
# # ==========================================
# def build_dummy_lb_df():
#     rows = []
#     for year, month in [(2025, 8), (2025, 9)]:
#         row = {'Month_Num': month, 'Year_Num': year}
#         row['month_sin'] = np.sin(2 * np.pi * month / 12)
#         row['month_cos'] = np.cos(2 * np.pi * month / 12)
#         row['Days_in_Month'] = calendar.monthrange(year, month)[1]
#         row['Public_Holidays'] = count_public_holidays(year, month)
#         row['Working_Days'] = count_working_days(year, month)
#         row['Quarter'] = (month - 1) // 3 + 1
#         row['Is_Year_End'] = 1 if month == 12 else 0
        
#         for city, df_weather in [('Jkt', monthly_rainfall_jkt), ('Sby', monthly_rainfall_sby)]:
#             rf = df_weather[(df_weather['year'] == year) & (df_weather['month'] == month)]
#             cols = ['Total_Rainfall_mm_', 'Rainy_Days_Count_', 'Max_Daily_Rainfall_', 'Avg_Daily_Rainfall_', 'Extreme_Rain_Days_']
#             for c in cols:
#                 row[c + city] = rf[c + city].values[0] if not rf.empty else 0
#         rows.append(row)
#     return pd.DataFrame(rows)

# def run_recursive_forecast_hp(models, df_train, df_future):
#     current_df = df_train.copy()
#     p_freq, p_sev, p_total = [], [], []

#     for i in range(len(df_future)):
#         f_row = df_future.iloc[i]
#         l_row = current_df.iloc[-1].copy()

#         for col in EXOG_COLS:
#             if col in f_row.index: 
#                 l_row[col] = f_row[col]
                
#         l_row['Time_Index'] = current_df.iloc[-1]['Time_Index'] + 1
#         l_row['Time_Index_Squared'] = l_row['Time_Index'] ** 2

#         for lag in range(1, 5):
#             l_row[f'Lag_{lag}_Total'] = current_df.iloc[-lag]['Total_Claim']
#             l_row[f'Lag_{lag}_Freq'] = current_df.iloc[-lag]['Claim_Frequency']
#             l_row[f'Lag_{lag}_Sev'] = current_df.iloc[-lag]['Claim_Severity']

#         for rm in [2, 3, 4]:
#             for t, src in [('Freq', 'Claim_Frequency'), ('Sev', 'Claim_Severity'), ('Total', 'Total_Claim')]:
#                 l_row[f'RM_{rm}_{t}'] = current_df.iloc[-rm:][src].mean()
#                 l_row[f'RStd_{rm}_{t}'] = current_df.iloc[-rm:][src].std()

#         l_row['Diff_1_Total'] = current_df.iloc[-1]['Total_Claim'] - current_df.iloc[-2]['Total_Claim']
#         l_row['Diff_1_Sev'] = current_df.iloc[-1]['Claim_Severity'] - current_df.iloc[-2]['Claim_Severity']
#         l_row['Diff_1_Freq'] = current_df.iloc[-1]['Claim_Frequency'] - current_df.iloc[-2]['Claim_Frequency']
#         l_row['Lag_1_Diff_1_Freq'] = current_df.iloc[-2]['Claim_Frequency'] - current_df.iloc[-3]['Claim_Frequency']
#         l_row['Acc_Freq'] = l_row['Diff_1_Freq'] - l_row['Lag_1_Diff_1_Freq']
#         l_row['Growth_Rate_Freq'] = (l_row['Diff_1_Freq'] / (l_row['Lag_2_Freq'] + 1e-6)) * 100
#         l_row['Momentum_2_Freq'] = l_row['Lag_1_Freq'] - l_row['Lag_3_Freq']
#         l_row['MACD_Freq'] = l_row['RM_2_Freq'] - l_row['RM_4_Freq']
        
#         past_3 = current_df.iloc[-3:]['Claim_Frequency']
#         l_row['Min_3_Freq'], l_row['Max_3_Freq'] = past_3.min(), past_3.max()
#         l_row['Stoch_K_Freq'] = (l_row['Lag_1_Freq'] - l_row['Min_3_Freq']) / (l_row['Max_3_Freq'] - l_row['Min_3_Freq'] + 1e-6)

#         if 'Total_Rainfall_mm_Jkt' in l_row:
#             l_row['Rain_x_LagFreq_Jkt'] = l_row['Total_Rainfall_mm_Jkt'] * l_row['Lag_1_Freq']
#         if 'Total_Rainfall_mm_Sby' in l_row:
#             l_row['Rain_x_LagFreq_Sby'] = l_row['Total_Rainfall_mm_Sby'] * l_row['Lag_1_Freq']

#         X_f = pd.DataFrame([l_row[feature_mapping['Claim_Frequency']]])
#         X_s = pd.DataFrame([l_row[feature_mapping['Claim_Severity']]])
        
#         pf = max(1e-6, models['Claim_Frequency'].predict(X_f)[0])
#         ps = max(1e-6, models['Claim_Severity'].predict(X_s)[0])
#         pt = pf * ps
        
#         p_freq.append(pf); p_sev.append(ps); p_total.append(pt)
        
#         new_entry = l_row.copy()
#         new_entry['Claim_Frequency'], new_entry['Claim_Severity'], new_entry['Total_Claim'] = pf, ps, pt
#         current_df = pd.concat([current_df, pd.DataFrame([new_entry])], ignore_index=True)
        
#     return {'Freq': p_freq, 'Sev': p_sev, 'Total': p_total}

# # ==========================================
# # 3. EVALUASI FITNESS
# # ==========================================
# def evaluate_fitness_hp(hp_freq, hp_sev, df_train):
#     try:
#         local_train, local_valid = df_train.iloc[:-5].copy(), df_train.iloc[-5:].copy()
        
#         # --- LOKAL ---
#         models_l = {}
#         for t, hp in [('Claim_Frequency', hp_freq), ('Claim_Severity', hp_sev)]:
#             # DIUBAH KE CatBoostRegressor
#             m = CatBoostRegressor(**hp, loss_function='MAPE', random_seed=42, verbose=0)
#             models_l[t] = m.fit(local_train[feature_mapping[t]], local_train[t])
            
#         res_l = run_recursive_forecast_hp(models_l, local_train, local_valid)
        
#         mape_l = (mean_absolute_percentage_error(local_valid['Total_Claim'], res_l['Total']) + 
#                   mean_absolute_percentage_error(local_valid['Claim_Frequency'], res_l['Freq']) + 
#                   mean_absolute_percentage_error(local_valid['Claim_Severity'], res_l['Sev'])) / 3 * 100

#         # --- KAGGLE LB ---
#         models_k = {}
#         for t, hp in [('Claim_Frequency', hp_freq), ('Claim_Severity', hp_sev)]:
#             # DIUBAH KE CatBoostRegressor
#             m = CatBoostRegressor(**hp, loss_function='MAPE', random_seed=42, verbose=0)
#             models_k[t] = m.fit(df_train[feature_mapping[t]], df_train[t])
            
#         df_test_lb = build_dummy_lb_df()
#         res_k = run_recursive_forecast_hp(models_k, df_train, df_test_lb)
        
#         base_f, base_s = [baseline[(2025,8)]['Freq'], baseline[(2025,9)]['Freq']], [baseline[(2025,8)]['Sev'], baseline[(2025,9)]['Sev']]
#         base_t = [f * s for f, s in zip(base_f, base_s)]
        
#         mape_k = (mean_absolute_percentage_error(base_t, res_k['Total']) + 
#                   mean_absolute_percentage_error(base_f, res_k['Freq']) + 
#                   mean_absolute_percentage_error(base_s, res_k['Sev'])) / 3 * 100

#         gap = abs(mape_l - mape_k)
        
#         # FITNESS DIAMBIL DARI LB DENGAN PENALTI GAP
#         fitness = mape_k if gap <= 2.0 else mape_k + 900.0 
        
#         return fitness, mape_l, mape_k
#     except Exception as e:
#         print(f"Error evaluasi: {e}")
#         return 999.0, 999.0, 999.0

# # ==========================================
# # 4. ENGINE EVOLUSI & EKSEKUSI
# # ==========================================
# def run_hp_evolution(df_train, pop_size=20, generations=10):
#     print(f"Mulai Tuning HP CatBoost (Populasi: {pop_size}, Generasi: {generations})")
    
#     population = [BEST_BASELINE_HP] + [
#         {
#             'Claim_Frequency': {p: random.choice(v) for p, v in HP_SPACE.items()}, 
#             'Claim_Severity': {p: random.choice(v) for p, v in HP_SPACE.items()}
#         } for _ in range(pop_size - 1)
#     ]
    
#     best_overall, best_lb_overall = None, 999.0
#     for gen in range(generations):
#         scored = []
#         for ind in population:
#             fit, loc, lb = evaluate_fitness_hp(ind['Claim_Frequency'], ind['Claim_Severity'], df_train)
#             scored.append({'fitness': fit, 'local': loc, 'lb': lb, 'dna': ind})
        
#         scored.sort(key=lambda x: x['fitness'])
#         if scored[0]['lb'] < best_lb_overall:
#             best_lb_overall, best_overall = scored[0]['lb'], scored[0]['dna']
            
#         print(f"MAPE: {scored[0]['local']:.4f}")
        
#         next_gen = [scored[0]['dna'], scored[1]['dna']]
#         while len(next_gen) < pop_size:
#             p1, p2 = random.choice(scored[:5])['dna'], random.choice(scored[:5])['dna']
#             child = {'Claim_Frequency': {}, 'Claim_Severity': {}}
#             for t in ['Claim_Frequency', 'Claim_Severity']:
#                 for p in HP_SPACE:
#                     val = random.choice([p1[t][p], p2[t][p]])
#                     if random.random() < 0.2: val = random.choice(HP_SPACE[p])
#                     child[t][p] = val
#             next_gen.append(child)
#         population = next_gen

#     print("\nTUNING SELESAI. HP TERBAIK:")
#     for t in ['Claim_Frequency', 'Claim_Severity']: print(f"{t}: {best_overall[t]}")
#     return best_overall

# # EKSEKUSI UTAMA
# monthly_data_clean = monthly_data_clean.sort_values("YearMonth").reset_index(drop=True)
# best_hp_final = run_hp_evolution(monthly_data_clean, pop_size=2, generations=5)

## EC Hyperparameter Tuning LGBM

In [ ]:
# feature_mapping = {
#     'Claim_Frequency': ['Lag_4_Total', 'Working_Days', 'RM_2_Freq', 'Time_Index_Squared'],
#     'Claim_Severity': ['Max_Daily_Rainfall_Jkt', 'Lag_2_Freq', 'month_cos', 'Lag_2_Total', 'Lag_4_Freq']
# }

# EXOG_COLS = ['Month_Num', 'Year_Num', 'month_sin', 'month_cos', 'Days_in_Month', 
#              'Working_Days', 'Quarter', 'Is_Year_End', 'Avg_Usia', 'Public_Holidays',
#              'Total_Rainfall_mm_Jkt', 'Max_Daily_Rainfall_Jkt', 'Avg_Daily_Rainfall_Jkt', 
#              'Extreme_Rain_Days_Jkt', 'Rainy_Days_Count_Jkt', 'Total_Rainfall_mm_Sby', 
#              'Max_Daily_Rainfall_Sby', 'Avg_Daily_Rainfall_Sby', 'Extreme_Rain_Days_Sby', 'Rainy_Days_Count_Sby']

# # BEST BASELINE DIUPDATE KE LGBM
# BEST_BASELINE_HP = {
#     'Claim_Frequency': {'n_estimators': 150, 'max_depth': 6, 'learning_rate': 0.08, 'reg_lambda': 5},
#     'Claim_Severity': {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.1, 'reg_lambda': 3}
# }

# HP_SPACE = {
#     'n_estimators': [50, 100, 150, 200, 250, 300],
#     'max_depth': [3, 4, 5, 6, 7],
#     'learning_rate': [0.01, 0.03, 0.05, 0.08, 0.1],
#     'reg_lambda': [1, 3, 5, 7, 10, 15]
# }

# # ==========================================
# # 2. HELPER & PREDIKSI REKURSIF
# # ==========================================
# def build_dummy_lb_df():
#     rows = []
#     for year, month in [(2025, 8), (2025, 9)]:
#         row = {'Month_Num': month, 'Year_Num': year}
#         row['month_sin'] = np.sin(2 * np.pi * month / 12)
#         row['month_cos'] = np.cos(2 * np.pi * month / 12)
#         row['Days_in_Month'] = calendar.monthrange(year, month)[1]
#         row['Public_Holidays'] = count_public_holidays(year, month)
#         row['Working_Days'] = count_working_days(year, month)
#         row['Quarter'] = (month - 1) // 3 + 1
#         row['Is_Year_End'] = 1 if month == 12 else 0
        
#         for city, df_weather in [('Jkt', monthly_rainfall_jkt), ('Sby', monthly_rainfall_sby)]:
#             rf = df_weather[(df_weather['year'] == year) & (df_weather['month'] == month)]
#             cols = ['Total_Rainfall_mm_', 'Rainy_Days_Count_', 'Max_Daily_Rainfall_', 'Avg_Daily_Rainfall_', 'Extreme_Rain_Days_']
#             for c in cols:
#                 row[c + city] = rf[c + city].values[0] if not rf.empty else 0
#         rows.append(row)
#     return pd.DataFrame(rows)

# def run_recursive_forecast_hp(models, df_train, df_future):
#     current_df = df_train.copy()
#     p_freq, p_sev, p_total = [], [], []

#     for i in range(len(df_future)):
#         f_row = df_future.iloc[i]
#         l_row = current_df.iloc[-1].copy()

#         for col in EXOG_COLS:
#             if col in f_row.index: 
#                 l_row[col] = f_row[col]
                
#         l_row['Time_Index'] = current_df.iloc[-1]['Time_Index'] + 1
#         l_row['Time_Index_Squared'] = l_row['Time_Index'] ** 2

#         for lag in range(1, 5):
#             l_row[f'Lag_{lag}_Total'] = current_df.iloc[-lag]['Total_Claim']
#             l_row[f'Lag_{lag}_Freq'] = current_df.iloc[-lag]['Claim_Frequency']
#             l_row[f'Lag_{lag}_Sev'] = current_df.iloc[-lag]['Claim_Severity']

#         for rm in [2, 3, 4]:
#             for t, src in [('Freq', 'Claim_Frequency'), ('Sev', 'Claim_Severity'), ('Total', 'Total_Claim')]:
#                 l_row[f'RM_{rm}_{t}'] = current_df.iloc[-rm:][src].mean()
#                 l_row[f'RStd_{rm}_{t}'] = current_df.iloc[-rm:][src].std()

#         l_row['Diff_1_Total'] = current_df.iloc[-1]['Total_Claim'] - current_df.iloc[-2]['Total_Claim']
#         l_row['Diff_1_Sev'] = current_df.iloc[-1]['Claim_Severity'] - current_df.iloc[-2]['Claim_Severity']
#         l_row['Diff_1_Freq'] = current_df.iloc[-1]['Claim_Frequency'] - current_df.iloc[-2]['Claim_Frequency']
#         l_row['Lag_1_Diff_1_Freq'] = current_df.iloc[-2]['Claim_Frequency'] - current_df.iloc[-3]['Claim_Frequency']
#         l_row['Acc_Freq'] = l_row['Diff_1_Freq'] - l_row['Lag_1_Diff_1_Freq']
#         l_row['Growth_Rate_Freq'] = (l_row['Diff_1_Freq'] / (l_row['Lag_2_Freq'] + 1e-6)) * 100
#         l_row['Momentum_2_Freq'] = l_row['Lag_1_Freq'] - l_row['Lag_3_Freq']
#         l_row['MACD_Freq'] = l_row['RM_2_Freq'] - l_row['RM_4_Freq']
        
#         past_3 = current_df.iloc[-3:]['Claim_Frequency']
#         l_row['Min_3_Freq'], l_row['Max_3_Freq'] = past_3.min(), past_3.max()
#         l_row['Stoch_K_Freq'] = (l_row['Lag_1_Freq'] - l_row['Min_3_Freq']) / (l_row['Max_3_Freq'] - l_row['Min_3_Freq'] + 1e-6)

#         if 'Total_Rainfall_mm_Jkt' in l_row:
#             l_row['Rain_x_LagFreq_Jkt'] = l_row['Total_Rainfall_mm_Jkt'] * l_row['Lag_1_Freq']
#         if 'Total_Rainfall_mm_Sby' in l_row:
#             l_row['Rain_x_LagFreq_Sby'] = l_row['Total_Rainfall_mm_Sby'] * l_row['Lag_1_Freq']

#         X_f = pd.DataFrame([l_row[feature_mapping['Claim_Frequency']]])
#         X_s = pd.DataFrame([l_row[feature_mapping['Claim_Severity']]])
        
#         pf = max(1e-6, models['Claim_Frequency'].predict(X_f)[0])
#         ps = max(1e-6, models['Claim_Severity'].predict(X_s)[0])
#         pt = pf * ps
        
#         p_freq.append(pf); p_sev.append(ps); p_total.append(pt)
        
#         new_entry = l_row.copy()
#         new_entry['Claim_Frequency'], new_entry['Claim_Severity'], new_entry['Total_Claim'] = pf, ps, pt
#         current_df = pd.concat([current_df, pd.DataFrame([new_entry])], ignore_index=True)
        
#     return {'Freq': p_freq, 'Sev': p_sev, 'Total': p_total}

# # ==========================================
# # 3. EVALUASI FITNESS
# # ==========================================
# def evaluate_fitness_hp(hp_freq, hp_sev, df_train):
#     try:
#         local_train, local_valid = df_train.iloc[:-5].copy(), df_train.iloc[-5:].copy()
        
#         # --- LOKAL ---
#         models_l = {}
#         for t, hp in [('Claim_Frequency', hp_freq), ('Claim_Severity', hp_sev)]:
#             # DIUBAH KE LGBMRegressor
#             m = LGBMRegressor(**hp, objective='mape', random_state=42, verbosity=-1, min_child_samples=1)
#             models_l[t] = m.fit(local_train[feature_mapping[t]], local_train[t])
            
#         res_l = run_recursive_forecast_hp(models_l, local_train, local_valid)
        
#         mape_l = (mean_absolute_percentage_error(local_valid['Total_Claim'], res_l['Total']) + 
#                   mean_absolute_percentage_error(local_valid['Claim_Frequency'], res_l['Freq']) + 
#                   mean_absolute_percentage_error(local_valid['Claim_Severity'], res_l['Sev'])) / 3 * 100

#         # --- KAGGLE LB ---
#         models_k = {}
#         for t, hp in [('Claim_Frequency', hp_freq), ('Claim_Severity', hp_sev)]:
#             # DIUBAH KE LGBMRegressor
#             m = LGBMRegressor(**hp, objective='mape', random_state=42, verbosity=-1, min_child_samples=1)
#             models_k[t] = m.fit(df_train[feature_mapping[t]], df_train[t])
            
#         df_test_lb = build_dummy_lb_df()
#         res_k = run_recursive_forecast_hp(models_k, df_train, df_test_lb)
        
#         base_f, base_s = [baseline[(2025,8)]['Freq'], baseline[(2025,9)]['Freq']], [baseline[(2025,8)]['Sev'], baseline[(2025,9)]['Sev']]
#         base_t = [f * s for f, s in zip(base_f, base_s)]
        
#         mape_k = (mean_absolute_percentage_error(base_t, res_k['Total']) + 
#                   mean_absolute_percentage_error(base_f, res_k['Freq']) + 
#                   mean_absolute_percentage_error(base_s, res_k['Sev'])) / 3 * 100

#         gap = abs(mape_l - mape_k)
        
#         # FITNESS DIAMBIL DARI LB DENGAN PENALTI GAP
#         fitness = mape_k if gap <= 2.0 else mape_k + 900.0 
        
#         return fitness, mape_l, mape_k
#     except Exception as e:
#         print(f"Error evaluasi: {e}")
#         return 999.0, 999.0, 999.0

# # ==========================================
# # 4. ENGINE EVOLUSI & EKSEKUSI
# # ==========================================
# def run_hp_evolution(df_train, pop_size=20, generations=10):
#     print(f"Mulai Tuning HP LightGBM (Populasi: {pop_size}, Generasi: {generations})")
    
#     population = [BEST_BASELINE_HP] + [
#         {
#             'Claim_Frequency': {p: random.choice(v) for p, v in HP_SPACE.items()}, 
#             'Claim_Severity': {p: random.choice(v) for p, v in HP_SPACE.items()}
#         } for _ in range(pop_size - 1)
#     ]
    
#     best_overall, best_lb_overall = None, 999.0
#     for gen in range(generations):
#         scored = []
#         for ind in population:
#             fit, loc, lb = evaluate_fitness_hp(ind['Claim_Frequency'], ind['Claim_Severity'], df_train)
#             scored.append({'fitness': fit, 'local': loc, 'lb': lb, 'dna': ind})
        
#         scored.sort(key=lambda x: x['fitness'])
#         if scored[0]['lb'] < best_lb_overall:
#             best_lb_overall, best_overall = scored[0]['lb'], scored[0]['dna']
            
#         print(f"Gen {gen+1} | FITNESS: {scored[0]['fitness']:.4f} | Lokal: {scored[0]['local']:.4f} | LB: {scored[0]['lb']:.4f}")
        
#         next_gen = [scored[0]['dna'], scored[1]['dna']]
#         while len(next_gen) < pop_size:
#             p1, p2 = random.choice(scored[:5])['dna'], random.choice(scored[:5])['dna']
#             child = {'Claim_Frequency': {}, 'Claim_Severity': {}}
#             for t in ['Claim_Frequency', 'Claim_Severity']:
#                 for p in HP_SPACE:
#                     val = random.choice([p1[t][p], p2[t][p]])
#                     if random.random() < 0.2: val = random.choice(HP_SPACE[p])
#                     child[t][p] = val
#             next_gen.append(child)
#         population = next_gen

#     print("\nTUNING SELESAI. HP TERBAIK:")
#     for t in ['Claim_Frequency', 'Claim_Severity']: print(f"{t}: {best_overall[t]}")
#     return best_overall

# # EKSEKUSI UTAMA
# monthly_data_clean = monthly_data_clean.sort_values("YearMonth").reset_index(drop=True)
# best_hp_final = run_hp_evolution(monthly_data_clean, pop_size=2, generations=5)

## EC Hyperparameter Tuning XGBoost

In [ ]:
# feature_mapping = {
#     'Claim_Frequency': ['Lag_4_Total', 'Working_Days', 'RM_2_Freq', 'Time_Index_Squared'],
#     'Claim_Severity': ['Max_Daily_Rainfall_Jkt', 'Lag_2_Freq', 'month_cos', 'Lag_2_Total', 'Lag_4_Freq']
# }

# EXOG_COLS = ['Month_Num', 'Year_Num', 'month_sin', 'month_cos', 'Days_in_Month', 
#              'Working_Days', 'Quarter', 'Is_Year_End', 'Avg_Usia', 'Public_Holidays',
#              'Total_Rainfall_mm_Jkt', 'Max_Daily_Rainfall_Jkt', 'Avg_Daily_Rainfall_Jkt', 
#              'Extreme_Rain_Days_Jkt', 'Rainy_Days_Count_Jkt', 'Total_Rainfall_mm_Sby', 
#              'Max_Daily_Rainfall_Sby', 'Avg_Daily_Rainfall_Sby', 'Extreme_Rain_Days_Sby', 'Rainy_Days_Count_Sby']

# BEST_BASELINE_HP = {
#     'Claim_Frequency': {'n_estimators': 50, 'max_depth': 3, 'learning_rate': 0.05, 'reg_lambda': 1},
#     'Claim_Severity': {'n_estimators': 250, 'max_depth': 4, 'learning_rate': 0.1, 'reg_lambda': 1}
# }

# HP_SPACE = {
#     'n_estimators': [50, 100, 150, 200, 250, 300],
#     'max_depth': [3, 4, 5, 6, 7],
#     'learning_rate': [0.01, 0.03, 0.05, 0.08, 0.1],
#     'reg_lambda': [1, 3, 5, 7, 10, 15]
# }

# # ==========================================
# # 2. HELPER & PREDIKSI REKURSIF
# # ==========================================
# def build_dummy_lb_df():
#     rows = []
#     for year, month in [(2025, 8), (2025, 9)]:
#         row = {'Month_Num': month, 'Year_Num': year}
#         row['month_sin'] = np.sin(2 * np.pi * month / 12)
#         row['month_cos'] = np.cos(2 * np.pi * month / 12)
#         row['Days_in_Month'] = calendar.monthrange(year, month)[1]
#         row['Public_Holidays'] = count_public_holidays(year, month)
#         row['Working_Days'] = count_working_days(year, month)
#         row['Quarter'] = (month - 1) // 3 + 1
#         row['Is_Year_End'] = 1 if month == 12 else 0
        
#         for city, df_weather in [('Jkt', monthly_rainfall_jkt), ('Sby', monthly_rainfall_sby)]:
#             rf = df_weather[(df_weather['year'] == year) & (df_weather['month'] == month)]
#             cols = ['Total_Rainfall_mm_', 'Rainy_Days_Count_', 'Max_Daily_Rainfall_', 'Avg_Daily_Rainfall_', 'Extreme_Rain_Days_']
#             for c in cols:
#                 row[c + city] = rf[c + city].values[0] if not rf.empty else 0
#         rows.append(row)
#     return pd.DataFrame(rows)

# def run_recursive_forecast_hp(models, df_train, df_future):
#     current_df = df_train.copy()
#     p_freq, p_sev, p_total = [], [], []

#     for i in range(len(df_future)):
#         f_row = df_future.iloc[i]
#         l_row = current_df.iloc[-1].copy()

#         for col in EXOG_COLS:
#             if col in f_row.index: 
#                 l_row[col] = f_row[col]
                
#         l_row['Time_Index'] = current_df.iloc[-1]['Time_Index'] + 1
#         l_row['Time_Index_Squared'] = l_row['Time_Index'] ** 2

#         for lag in range(1, 5):
#             l_row[f'Lag_{lag}_Total'] = current_df.iloc[-lag]['Total_Claim']
#             l_row[f'Lag_{lag}_Freq'] = current_df.iloc[-lag]['Claim_Frequency']
#             l_row[f'Lag_{lag}_Sev'] = current_df.iloc[-lag]['Claim_Severity']

#         for rm in [2, 3, 4]:
#             for t, src in [('Freq', 'Claim_Frequency'), ('Sev', 'Claim_Severity'), ('Total', 'Total_Claim')]:
#                 l_row[f'RM_{rm}_{t}'] = current_df.iloc[-rm:][src].mean()
#                 l_row[f'RStd_{rm}_{t}'] = current_df.iloc[-rm:][src].std()

#         l_row['Diff_1_Total'] = current_df.iloc[-1]['Total_Claim'] - current_df.iloc[-2]['Total_Claim']
#         l_row['Diff_1_Sev'] = current_df.iloc[-1]['Claim_Severity'] - current_df.iloc[-2]['Claim_Severity']
#         l_row['Diff_1_Freq'] = current_df.iloc[-1]['Claim_Frequency'] - current_df.iloc[-2]['Claim_Frequency']
#         l_row['Lag_1_Diff_1_Freq'] = current_df.iloc[-2]['Claim_Frequency'] - current_df.iloc[-3]['Claim_Frequency']
#         l_row['Acc_Freq'] = l_row['Diff_1_Freq'] - l_row['Lag_1_Diff_1_Freq']
#         l_row['Growth_Rate_Freq'] = (l_row['Diff_1_Freq'] / (l_row['Lag_2_Freq'] + 1e-6)) * 100
#         l_row['Momentum_2_Freq'] = l_row['Lag_1_Freq'] - l_row['Lag_3_Freq']
#         l_row['MACD_Freq'] = l_row['RM_2_Freq'] - l_row['RM_4_Freq']
        
#         past_3 = current_df.iloc[-3:]['Claim_Frequency']
#         l_row['Min_3_Freq'], l_row['Max_3_Freq'] = past_3.min(), past_3.max()
#         l_row['Stoch_K_Freq'] = (l_row['Lag_1_Freq'] - l_row['Min_3_Freq']) / (l_row['Max_3_Freq'] - l_row['Min_3_Freq'] + 1e-6)

#         if 'Total_Rainfall_mm_Jkt' in l_row:
#             l_row['Rain_x_LagFreq_Jkt'] = l_row['Total_Rainfall_mm_Jkt'] * l_row['Lag_1_Freq']
#         if 'Total_Rainfall_mm_Sby' in l_row:
#             l_row['Rain_x_LagFreq_Sby'] = l_row['Total_Rainfall_mm_Sby'] * l_row['Lag_1_Freq']

#         X_f = pd.DataFrame([l_row[feature_mapping['Claim_Frequency']]])
#         X_s = pd.DataFrame([l_row[feature_mapping['Claim_Severity']]])
        
#         pf = max(1e-6, models['Claim_Frequency'].predict(X_f)[0])
#         ps = max(1e-6, models['Claim_Severity'].predict(X_s)[0])
#         pt = pf * ps
        
#         p_freq.append(pf); p_sev.append(ps); p_total.append(pt)
        
#         new_entry = l_row.copy()
#         new_entry['Claim_Frequency'], new_entry['Claim_Severity'], new_entry['Total_Claim'] = pf, ps, pt
#         current_df = pd.concat([current_df, pd.DataFrame([new_entry])], ignore_index=True)
        
#     return {'Freq': p_freq, 'Sev': p_sev, 'Total': p_total}

# # ==========================================
# # 3. EVALUASI FITNESS
# # ==========================================
# def evaluate_fitness_hp(hp_freq, hp_sev, df_train):
#     try:
#         local_train, local_valid = df_train.iloc[:-5].copy(), df_train.iloc[-5:].copy()
        
#         # --- LOKAL ---
#         models_l = {}
#         for t, hp in [('Claim_Frequency', hp_freq), ('Claim_Severity', hp_sev)]:
#             m = XGBRegressor(**hp, objective='reg:absoluteerror', random_state=42, verbosity=0)
#             models_l[t] = m.fit(local_train[feature_mapping[t]], local_train[t])
            
#         res_l = run_recursive_forecast_hp(models_l, local_train, local_valid)
        
#         mape_l = (mean_absolute_percentage_error(local_valid['Total_Claim'], res_l['Total']) + 
#                   mean_absolute_percentage_error(local_valid['Claim_Frequency'], res_l['Freq']) + 
#                   mean_absolute_percentage_error(local_valid['Claim_Severity'], res_l['Sev'])) / 3 * 100

#         # --- KAGGLE LB ---
#         models_k = {}
#         for t, hp in [('Claim_Frequency', hp_freq), ('Claim_Severity', hp_sev)]:
#             m = XGBRegressor(**hp, objective='reg:absoluteerror', random_state=42, verbosity=0)
#             models_k[t] = m.fit(df_train[feature_mapping[t]], df_train[t])
            
#         df_test_lb = build_dummy_lb_df()
#         res_k = run_recursive_forecast_hp(models_k, df_train, df_test_lb)
        
#         base_f, base_s = [baseline[(2025,8)]['Freq'], baseline[(2025,9)]['Freq']], [baseline[(2025,8)]['Sev'], baseline[(2025,9)]['Sev']]
#         base_t = [f * s for f, s in zip(base_f, base_s)]
        
#         mape_k = (mean_absolute_percentage_error(base_t, res_k['Total']) + 
#                   mean_absolute_percentage_error(base_f, res_k['Freq']) + 
#                   mean_absolute_percentage_error(base_s, res_k['Sev'])) / 3 * 100

#         gap = abs(mape_l - mape_k)
        
#         fitness = mape_k if gap <= 2.0 else mape_k + 900.0 
        
#         return fitness, mape_l, mape_k
#     except Exception as e:
#         print(f"Error evaluasi: {e}")
#         return 999.0, 999.0, 999.0

# # ==========================================
# # 4. ENGINE EVOLUSI & EKSEKUSI
# # ==========================================
# def run_hp_evolution(df_train, pop_size=20, generations=10):
#     print(f"Mulai Tuning HP XGBoost (Populasi: {pop_size}, Generasi: {generations})")
    
#     population = [BEST_BASELINE_HP] + [
#         {
#             'Claim_Frequency': {p: random.choice(v) for p, v in HP_SPACE.items()}, 
#             'Claim_Severity': {p: random.choice(v) for p, v in HP_SPACE.items()}
#         } for _ in range(pop_size - 1)
#     ]
    
#     best_overall, best_lb_overall = None, 999.0
#     for gen in range(generations):
#         scored = []
#         for ind in population:
#             fit, loc, lb = evaluate_fitness_hp(ind['Claim_Frequency'], ind['Claim_Severity'], df_train)
#             scored.append({'fitness': fit, 'local': loc, 'lb': lb, 'dna': ind})
        
#         scored.sort(key=lambda x: x['fitness'])
#         if scored[0]['lb'] < best_lb_overall:
#             best_lb_overall, best_overall = scored[0]['lb'], scored[0]['dna']
            
#         print(f"MAPE: {scored[0]['local']:.4f} ")
        
#         next_gen = [scored[0]['dna'], scored[1]['dna']]
#         while len(next_gen) < pop_size:
#             p1, p2 = random.choice(scored[:5])['dna'], random.choice(scored[:5])['dna']
#             child = {'Claim_Frequency': {}, 'Claim_Severity': {}}
#             for t in ['Claim_Frequency', 'Claim_Severity']:
#                 for p in HP_SPACE:
#                     val = random.choice([p1[t][p], p2[t][p]])
#                     if random.random() < 0.2: val = random.choice(HP_SPACE[p])
#                     child[t][p] = val
#             next_gen.append(child)
#         population = next_gen

#     print("\nTUNING SELESAI. HP TERBAIK:")
#     for t in ['Claim_Frequency', 'Claim_Severity']: print(f"{t}: {best_overall[t]}")
#     return best_overall

# # EKSEKUSI UTAMA
# monthly_data_clean = monthly_data_clean.sort_values("YearMonth").reset_index(drop=True)
# best_hp_final = run_hp_evolution(monthly_data_clean, pop_size=2, generations=15)

# Train Model Total Claim Agregat

## Feature Selection

In [ ]:
feature_mapping = {
    'Claim_Frequency': ['Lag_4_Total', 'Working_Days', 'RM_2_Freq', 'Time_Index_Squared'],


    'Claim_Severity' : ['Max_Daily_Rainfall_Jkt', 'Lag_2_Freq', 'month_cos', 'Lag_2_Total', 'Lag_4_Freq']
}
target_cols = ['Total_Claim', 'Claim_Frequency', 'Claim_Severity']

## Traning Total Claim Agregat

In [ ]:
monthly_data_clean = monthly_data_clean.sort_values("YearMonth").reset_index(drop=True)
main_targets = ['Claim_Frequency', 'Claim_Severity']
valid_size = 5

train_df = monthly_data_clean.iloc[:-valid_size].copy()
valid_df = monthly_data_clean.iloc[-valid_size:].copy()

# Konfigurasi Hiperparameter Hasil Evolusi
best_hp_configs = {
    'CatBoost': {
        'Claim_Frequency': {'iterations': 200, 'depth': 3, 'learning_rate': 0.08, 'l2_leaf_reg': 5},
        'Claim_Severity': {'iterations': 300, 'depth': 4, 'learning_rate': 0.08, 'l2_leaf_reg': 15}
    },
    'LGBM': {
        'Claim_Frequency': {'n_estimators': 150, 'max_depth': 6, 'learning_rate': 0.08, 'reg_lambda': 5},
        'Claim_Severity': {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.1, 'reg_lambda': 3}

    },
    'XGBoost': {
        'Claim_Frequency': {'n_estimators': 50, 'max_depth': 3, 'learning_rate': 0.05, 'reg_lambda': 1},
        'Claim_Severity': {'n_estimators': 250, 'max_depth': 4, 'learning_rate': 0.1, 'reg_lambda': 1}
    }
}

# 1. Pelatihan Model
final_models = {fw: {} for fw in best_hp_configs.keys()}
for fw, targets in best_hp_configs.items():
    for target, hp in targets.items():
        X_train, y_train = train_df[feature_mapping[target]], train_df[target]
        if fw == 'CatBoost':
            m = CatBoostRegressor(**hp, loss_function='MAPE', random_seed=42, verbose=0)
        elif fw == 'LGBM':
            m = LGBMRegressor(**hp, objective='mape', random_state=42, verbosity=-1, min_child_samples=1)
        else:
            m = XGBRegressor(**hp, objective='reg:absoluteerror', random_state=42)
        final_models[fw][target] = m.fit(X_train, y_train)

# 2. Prediksi Rekursif
results_mape = {}
exog = ['Month_Num', 'Year_Num', 'month_sin', 'month_cos', 'Days_in_Month', 
        'Working_Days', 'Quarter', 'Is_Year_End', 'Avg_Usia', 'Public_Holidays',
        'Total_Rainfall_mm_Jkt', 'Max_Daily_Rainfall_Jkt', 'Avg_Daily_Rainfall_Jkt', 
        'Extreme_Rain_Days_Jkt', 'Rainy_Days_Count_Jkt', 'Total_Rainfall_mm_Sby', 
        'Max_Daily_Rainfall_Sby', 'Avg_Daily_Rainfall_Sby', 'Extreme_Rain_Days_Sby', 'Rainy_Days_Count_Sby']

for fw in best_hp_configs.keys():
    current_df = train_df.copy()
    p_freq, p_sev, p_total = [], [], []

    for i in range(valid_size):
        f_row = valid_df.iloc[i]
        l_row = current_df.iloc[-1].copy()

        # Update Fitur
        for col in exog:
            if col in f_row.index: l_row[col] = f_row[col]
            
        l_row['Time_Index'] = current_df.iloc[-1]['Time_Index'] + 1
        l_row['Time_Index_Squared'] = l_row['Time_Index'] ** 2

        for lag in range(1, 5):
            l_row[f'Lag_{lag}_Total'] = current_df.iloc[-lag]['Total_Claim']
            l_row[f'Lag_{lag}_Freq'] = current_df.iloc[-lag]['Claim_Frequency']
            l_row[f'Lag_{lag}_Sev'] = current_df.iloc[-lag]['Claim_Severity']

        for rm in [2, 3, 4]:
            for t, src in [('Freq', 'Claim_Frequency'), ('Sev', 'Claim_Severity'), ('Total', 'Total_Claim')]:
                l_row[f'RM_{rm}_{t}'] = current_df.iloc[-rm:][src].mean()
                l_row[f'RStd_{rm}_{t}'] = current_df.iloc[-rm:][src].std()

        l_row['Diff_1_Total'] = current_df.iloc[-1]['Total_Claim'] - current_df.iloc[-2]['Total_Claim']
        l_row['Diff_1_Sev'] = current_df.iloc[-1]['Claim_Severity'] - current_df.iloc[-2]['Claim_Severity']
        l_row['Diff_1_Freq'] = current_df.iloc[-1]['Claim_Frequency'] - current_df.iloc[-2]['Claim_Frequency']
        l_row['Lag_1_Diff_1_Freq'] = current_df.iloc[-2]['Claim_Frequency'] - current_df.iloc[-3]['Claim_Frequency']
        l_row['Acc_Freq'] = l_row['Diff_1_Freq'] - l_row['Lag_1_Diff_1_Freq']
        l_row['Growth_Rate_Freq'] = (l_row['Diff_1_Freq'] / (l_row['Lag_2_Freq'] + 1e-6)) * 100
        l_row['Momentum_2_Freq'] = l_row['Lag_1_Freq'] - l_row['Lag_3_Freq']
        l_row['MACD_Freq'] = l_row['RM_2_Freq'] - l_row['RM_4_Freq']
        
        past_3 = current_df.iloc[-3:]['Claim_Frequency']
        l_row['Min_3_Freq'], l_row['Max_3_Freq'] = past_3.min(), past_3.max()
        l_row['Stoch_K_Freq'] = (l_row['Lag_1_Freq'] - l_row['Min_3_Freq']) / (l_row['Max_3_Freq'] - l_row['Min_3_Freq'] + 1e-6)

        if 'Total_Rainfall_mm_Jkt' in l_row:
            l_row['Rain_x_LagFreq_Jkt'] = l_row['Total_Rainfall_mm_Jkt'] * l_row['Lag_1_Freq']
        if 'Total_Rainfall_mm_Sby' in l_row:
            l_row['Rain_x_LagFreq_Sby'] = l_row['Total_Rainfall_mm_Sby'] * l_row['Lag_1_Freq']

        # Prediksi
        pf = max(1e-6, final_models[fw]['Claim_Frequency'].predict(pd.DataFrame([l_row[feature_mapping['Claim_Frequency']]]))[0])
        ps = max(1e-6, final_models[fw]['Claim_Severity'].predict(pd.DataFrame([l_row[feature_mapping['Claim_Severity']]]))[0])
        pt = pf * ps
        
        p_freq.append(pf); p_sev.append(ps); p_total.append(pt)
        new_entry = l_row.copy()
        new_entry['Claim_Frequency'], new_entry['Claim_Severity'], new_entry['Total_Claim'] = pf, ps, pt
        current_df = pd.concat([current_df, pd.DataFrame([new_entry])], ignore_index=True)

    results_mape[fw] = {
        'Total': mean_absolute_percentage_error(valid_df['Total_Claim'], p_total),
        'Freq': mean_absolute_percentage_error(valid_df['Claim_Frequency'], p_freq),
        'Sev': mean_absolute_percentage_error(valid_df['Claim_Severity'], p_sev)
    }

# 3. Output
for fw, m in results_mape.items():
    avg = (m['Total'] + m['Freq'] + m['Sev']) / 3
    print(f"\n--- HASIL VALIDASI REKURSIF {fw.upper()} ---")
    print(f"Total_Claim MAPE: {round(m['Total']*100, 4)}% | Freq MAPE: {round(m['Freq']*100, 4)}% | Sev MAPE: {round(m['Sev']*100, 4)}%")
    print(f"FINAL AVG SCORE: {round(avg * 100, 4)}%")

## Feature Importance

In [ ]:
# 1. Pengumpulan dan Normalisasi Data Lintas Framework
frameworks = ['CatBoost', 'LGBM', 'XGBoost']
targets = ['Claim_Frequency', 'Claim_Severity']
all_imp_data = []

for fw in frameworks:
    for target in targets:
        model = final_models[fw][target]
        feats = feature_mapping[target]
        
        # Ekstraksi skor mentah
        scores = model.get_feature_importance() if fw == 'CatBoost' else model.feature_importances_
        
        # Normalisasi ke skala 100%
        total = sum(scores) + 1e-10
        for f, s in zip(feats, scores):
            all_imp_data.append({
                'Framework': fw,
                'Target': target,
                'Feature': f,
                'Importance (%)': (s / total) * 100
            })

df_all_imp = pd.DataFrame(all_imp_data)

# 2. Plotting Grafik Terpisah Menggunakan Clustered Column
sns.set_theme(style="whitegrid")

for target in targets:
    plt.figure(figsize=(14, 6))
    df_subset = df_all_imp[df_all_imp['Target'] == target]
    
    sns.barplot(
        data=df_subset,
        x="Feature",
        y="Importance (%)",
        hue="Framework",
        palette="viridis"
    )
    
    plt.title(f"Perbandingan Feature Importance: {target}", fontsize=14)
    plt.xlabel("Nama Fitur", fontsize=12)
    plt.ylabel("Kontribusi Relatif (%)", fontsize=12)
    plt.xticks(rotation=45)
    plt.legend(title="Framework", loc="upper right")
    plt.tight_layout()
    plt.show()

## Traning 100% Total Agregat

In [ ]:
final_models = {}

# Simpan hyperparameter hasil EC
best_hp = {
   'Claim_Frequency': {'iterations': 200, 'depth': 3, 'learning_rate': 0.08, 'l2_leaf_reg': 5},
   'Claim_Severity': {'iterations': 300, 'depth': 4, 'learning_rate': 0.08, 'l2_leaf_reg': 15}
   
}

target_cols = ['Claim_Frequency', 'Claim_Severity']

for target in target_cols:
    
    features_to_use = feature_mapping[target]
    X_full = monthly_data_clean[features_to_use]
    y_full = monthly_data_clean[target]
    
    print(f"Melatih model final untuk: {target} (Fitur: {len(features_to_use)})")
    
    # Ambil parameter spesifik untuk target yang sedang dilatih
    params = best_hp[target]
    
    model = CatBoostRegressor(
        iterations=params['iterations'],
        depth=params['depth'],
        learning_rate=params['learning_rate'],
        l2_leaf_reg=params['l2_leaf_reg'],
        loss_function='MAPE',
        random_seed=42,
        verbose=0,
        allow_writing_files=False
    )
    
    # Fit model pada seluruh data
    model.fit(X_full, y_full)
    
    final_models[target] = model

print("Pelatihan model Frequency & Severity selesai.")

## Save Model

In [ ]:

for target, model in final_models.items():
    file_path = f"Tambahan/model_{target.lower()}.joblib"
    joblib.dump(model, file_path)
    print(f"Model {target} berhasil disimpan pada: {file_path}")

print("Proses penyimpanan seluruh model dengan format joblib selesai.")

## Feature Importance

In [ ]:
target_cols = ['Claim_Frequency', 'Claim_Severity']

all_importances = []

print(f"\nMenghitung Feature Importance Final untuk {target_cols}...")

# 2. Looping khusus untuk 2 target tersebut
for target in target_cols:
    features = feature_mapping[target]
    
    # Ambil nilai feature importance langsung dari model final yang udah di-train
    importance_scores = final_models[target].get_feature_importance()
    
    for feat, score in zip(features, importance_scores):
        all_importances.append({
            'Target': target,
            'Feature': feat,
            'Importance Score': round(score, 4)
        })

# 3. Convert ke DataFrame
df_feature_importance_final = pd.DataFrame(all_importances)

# 4. Urutkan berdasarkan Target dan Score tertinggi ke terendah
df_feature_importance_final = df_feature_importance_final.sort_values(
    by=['Target', 'Importance Score'], 
    ascending=[True, False]
)

# 5. Tampilkan hasilnya secara utuh
print("\nTABEL FINAL FEATURE IMPORTANCE (MODEL 100% DATASET)")
pd.set_option('display.max_rows', None) 
display(df_feature_importance_final)
pd.reset_option('display.max_rows')

## Forecast Until 2025 Dec

In [ ]:
forecast_months = [(2025, 8), (2025, 9), (2025, 10), (2025, 11), (2025, 12)]
current_df = monthly_data_clean.copy()
submission_data = []

print("Memulai forecasting rekursif dengan proteksi freezing...")

for year, month in forecast_months:
    # A. Inisialisasi baris baru dari baris terakhir
    last_row = current_df.iloc[-1].copy()

    # B. Update Fitur Eksogen (Waktu & Kalender)
    last_row['Month_Num'] = month
    last_row['Year_Num'] = year
    last_row['month_sin'] = np.sin(2 * np.pi * month / 12)
    last_row['month_cos'] = np.cos(2 * np.pi * month / 12)
    last_row['Days_in_Month'] = calendar.monthrange(year, month)[1]
    last_row['Public_Holidays'] = count_public_holidays(year, month)
    last_row['Working_Days'] = count_working_days(year, month)
    last_row['Quarter'] = (month - 1) // 3 + 1
    last_row['Is_Year_End'] = 1 if month == 12 else 0
    
    # C. Update Fitur Eksogen (Cuaca JKT & SBY)
    for city, df_weather in [('Jkt', monthly_rainfall_jkt), ('Sby', monthly_rainfall_sby)]:
        rf = df_weather[(df_weather['year'] == year) & (df_weather['month'] == month)]
        cols = ['Total_Rainfall_mm_', 'Rainy_Days_Count_', 'Max_Daily_Rainfall_', 'Avg_Daily_Rainfall_', 'Extreme_Rain_Days_']
        if not rf.empty:
            for c in cols:
                last_row[c + city] = rf[c + city].values[0]
        else:
            for c in cols:
                last_row[c + city] = 0
        
    # D. Update Time Index
    last_row['Time_Index'] = current_df.iloc[-1]['Time_Index'] + 1
    last_row['Time_Index_Squared'] = last_row['Time_Index'] ** 2
    
    # E. Update Lags (1-4) untuk Freq, Sev, dan Total
    for lag in range(1, 5):
        last_row[f'Lag_{lag}_Freq'] = current_df.iloc[-lag]['Claim_Frequency']
        last_row[f'Lag_{lag}_Sev'] = current_df.iloc[-lag]['Claim_Severity']
        last_row[f'Lag_{lag}_Total'] = current_df.iloc[-lag]['Total_Claim']

    # F. Update Diferensiasi (Diffs)
    last_row['Diff_1_Freq'] = current_df.iloc[-1]['Claim_Frequency'] - current_df.iloc[-2]['Claim_Frequency']
    last_row['Diff_1_Sev'] = current_df.iloc[-1]['Claim_Severity'] - current_df.iloc[-2]['Claim_Severity']
    last_row['Diff_1_Total'] = current_df.iloc[-1]['Total_Claim'] - current_df.iloc[-2]['Total_Claim']
    
    # G. Update Akselerasi (Diferensiasi Tingkat 2)
    last_row['Lag_1_Diff_1_Freq'] = current_df.iloc[-2]['Claim_Frequency'] - current_df.iloc[-3]['Claim_Frequency']
    last_row['Acc_Freq'] = last_row['Diff_1_Freq'] - last_row['Lag_1_Diff_1_Freq']

    # H. Update Rolling Statistics (Mean & Std)
    for rm in [2, 3, 4]:
        for t, col_name in [('Freq', 'Claim_Frequency'), ('Sev', 'Claim_Severity'), ('Total', 'Total_Claim')]:
            last_row[f'RM_{rm}_{t}'] = current_df.iloc[-rm:][col_name].mean()
            last_row[f'RStd_{rm}_{t}'] = current_df.iloc[-rm:][col_name].std()

    # I. Update Technical Indicators (Z-Score, MACD, Stoch, Momentum)
    last_row['Min_3_Freq'] = current_df.iloc[-3:]['Claim_Frequency'].min()
    last_row['Max_3_Freq'] = current_df.iloc[-3:]['Claim_Frequency'].max()
    last_row['MACD_Freq'] = last_row['RM_2_Freq'] - last_row['RM_4_Freq']
    last_row['Z_Score_Freq'] = (last_row['Lag_1_Freq'] - last_row['RM_4_Freq']) / (last_row['RStd_4_Freq'] + 1e-6)
    last_row['Stoch_K_Freq'] = (last_row['Lag_1_Freq'] - last_row['Min_3_Freq']) / (last_row['Max_3_Freq'] - last_row['Min_3_Freq'] + 1e-6)
    last_row['Momentum_2_Freq'] = last_row['Lag_1_Freq'] - last_row['Lag_3_Freq']
    last_row['Growth_Rate_Freq'] = (last_row['Diff_1_Freq'] / (last_row['Lag_2_Freq'] + 1e-6)) * 100

    # J. Update Interaksi (Rain x Lag)
    last_row['Rain_x_LagFreq_Jkt'] = last_row['Total_Rainfall_mm_Jkt'] * last_row['Lag_1_Freq']
    last_row['Rain_x_LagFreq_Sby'] = last_row['Total_Rainfall_mm_Sby'] * last_row['Lag_1_Freq']

    # K. PREDIKSI (Hanya memanggil fitur yang ada di mapping)
    X_f = pd.DataFrame([last_row[feature_mapping['Claim_Frequency']]])
    pred_f = max(1, final_models['Claim_Frequency'].predict(X_f)[0])

    X_s = pd.DataFrame([last_row[feature_mapping['Claim_Severity']]])
    pred_s = max(1e-6, final_models['Claim_Severity'].predict(X_s)[0])

    pred_t = pred_f * pred_s

    # L. Finalisasi Baris dan Update current_df
    new_entry = last_row.copy()
    new_entry['Claim_Frequency'], new_entry['Claim_Severity'], new_entry['Total_Claim'] = pred_f, pred_s, pred_t
    current_df = pd.concat([current_df, pd.DataFrame([new_entry])], ignore_index=True)

    # M. Simpan Data Submission
    m_id = f"{year}_{month:02d}"
    submission_data.extend([
        {'id': f"{m_id}_Claim_Frequency", 'value': pred_f},
        {'id': f"{m_id}_Claim_Severity", 'value': pred_s},
        {'id': f"{m_id}_Total_Claim", 'value': pred_t}
    ])

print("Forecasting rekursif selesai dengan fitur yang sepenuhnya dinamis.")

## Submission Kaggle

In [ ]:

df_submission_final = pd.DataFrame(submission_data)
filename = 'Tambahan/submission_kaggle.csv'

df_submission_final.to_csv(filename, index=False)

print(f"File berhasil disimpan: {filename}")
print("Preview hasil akhir:")
display(df_submission_final)

## Forecast Until 2026

In [ ]:
# 1. Definisi kolom cuaca
cols_jkt = ['Total_Rainfall_mm_Jkt', 'Avg_Daily_Rainfall_Jkt', 'Max_Daily_Rainfall_Jkt', 'Rainy_Days_Count_Jkt', 'Extreme_Rain_Days_Jkt']
cols_sby = ['Total_Rainfall_mm_Sby', 'Avg_Daily_Rainfall_Sby', 'Max_Daily_Rainfall_Sby', 'Rainy_Days_Count_Sby', 'Extreme_Rain_Days_Sby']

# 2. Buat profil bulanan (seasonal mean) dari monthly_data_clean
# Ini akan digunakan jika data cuaca aktual tahun 2026 tidak tersedia
weather_profile_jkt = monthly_data_clean.groupby('Month_Num')[cols_jkt].mean().to_dict('index')
weather_profile_sby = monthly_data_clean.groupby('Month_Num')[cols_sby].mean().to_dict('index')

print("Profil cuaca bulanan berhasil dibuat untuk fallback tahun 2026.")

In [ ]:
forecast_months = [(2025, 8), (2025, 9), (2025, 10), (2025, 11), (2025, 12)]
forecast_months += [(2026, m) for m in range(1, 13)]
current_df = monthly_data_clean.copy()
submission_data = []

print("Memulai forecasting rekursif dengan proteksi freezing...")

for year, month in forecast_months:
    # A. Inisialisasi baris baru dari baris terakhir
    last_row = current_df.iloc[-1].copy()

    # B. Update Fitur Eksogen (Waktu & Kalender)
    last_row['Month_Num'], last_row['Year_Num'] = month, year
    last_row['Year_Num'] = year
    last_row['month_sin'] = np.sin(2 * np.pi * month / 12)
    last_row['month_cos'] = np.cos(2 * np.pi * month / 12)
    last_row['Days_in_Month'] = calendar.monthrange(year, month)[1]
    last_row['Public_Holidays'] = count_public_holidays(year, month)
    last_row['Working_Days'] = count_working_days(year, month)
    last_row['Quarter'] = (month - 1) // 3 + 1
    last_row['Is_Year_End'] = 1 if month == 12 else 0
    
    # C. Update Fitur Eksogen (Cuaca JKT & SBY)
    for city, df_weather, profile in [('Jkt', monthly_rainfall_jkt, weather_profile_jkt), 
                                      ('Sby', monthly_rainfall_sby, weather_profile_sby)]:
        
        # Cari data aktual
        rf = df_weather[(df_weather['year'] == year) & (df_weather['month'] == month)]
        cols = ['Total_Rainfall_mm_', 'Rainy_Days_Count_', 'Max_Daily_Rainfall_', 'Avg_Daily_Rainfall_', 'Extreme_Rain_Days_']
        
        if not rf.empty:
            # Jika ada data (untuk 2025)
            for c in cols: last_row[c + city] = rf[c + city].values[0]
        else:
            # Jika data tidak ada (untuk 2026), ambil dari profil rata-rata historis
            for c in cols: last_row[c + city] = profile[month][c + city]
        
    # D. Update Time Index
    last_row['Time_Index'] = current_df.iloc[-1]['Time_Index'] + 1
    last_row['Time_Index_Squared'] = last_row['Time_Index'] ** 2
    
    # E. Update Lags (1-4) untuk Freq, Sev, dan Total
    for lag in range(1, 5):
        last_row[f'Lag_{lag}_Freq'] = current_df.iloc[-lag]['Claim_Frequency']
        last_row[f'Lag_{lag}_Sev'] = current_df.iloc[-lag]['Claim_Severity']
        last_row[f'Lag_{lag}_Total'] = current_df.iloc[-lag]['Total_Claim']

    # F. Update Diferensiasi (Diffs)
    last_row['Diff_1_Freq'] = current_df.iloc[-1]['Claim_Frequency'] - current_df.iloc[-2]['Claim_Frequency']
    last_row['Diff_1_Sev'] = current_df.iloc[-1]['Claim_Severity'] - current_df.iloc[-2]['Claim_Severity']
    last_row['Diff_1_Total'] = current_df.iloc[-1]['Total_Claim'] - current_df.iloc[-2]['Total_Claim']
    
    # G. Update Akselerasi (Diferensiasi Tingkat 2)
    last_row['Lag_1_Diff_1_Freq'] = current_df.iloc[-2]['Claim_Frequency'] - current_df.iloc[-3]['Claim_Frequency']
    last_row['Acc_Freq'] = last_row['Diff_1_Freq'] - last_row['Lag_1_Diff_1_Freq']

    # H. Update Rolling Statistics (Mean & Std)
    for rm in [2, 3, 4]:
        for t, col_name in [('Freq', 'Claim_Frequency'), ('Sev', 'Claim_Severity'), ('Total', 'Total_Claim')]:
            last_row[f'RM_{rm}_{t}'] = current_df.iloc[-rm:][col_name].mean()
            last_row[f'RStd_{rm}_{t}'] = current_df.iloc[-rm:][col_name].std()

    # I. Update Technical Indicators (Z-Score, MACD, Stoch, Momentum)
    last_row['Min_3_Freq'] = current_df.iloc[-3:]['Claim_Frequency'].min()
    last_row['Max_3_Freq'] = current_df.iloc[-3:]['Claim_Frequency'].max()
    last_row['MACD_Freq'] = last_row['RM_2_Freq'] - last_row['RM_4_Freq']
    last_row['Z_Score_Freq'] = (last_row['Lag_1_Freq'] - last_row['RM_4_Freq']) / (last_row['RStd_4_Freq'] + 1e-6)
    last_row['Stoch_K_Freq'] = (last_row['Lag_1_Freq'] - last_row['Min_3_Freq']) / (last_row['Max_3_Freq'] - last_row['Min_3_Freq'] + 1e-6)
    last_row['Momentum_2_Freq'] = last_row['Lag_1_Freq'] - last_row['Lag_3_Freq']
    last_row['Growth_Rate_Freq'] = (last_row['Diff_1_Freq'] / (last_row['Lag_2_Freq'] + 1e-6)) * 100

    # J. Update Interaksi (Rain x Lag)
    last_row['Rain_x_LagFreq_Jkt'] = last_row['Total_Rainfall_mm_Jkt'] * last_row['Lag_1_Freq']
    last_row['Rain_x_LagFreq_Sby'] = last_row['Total_Rainfall_mm_Sby'] * last_row['Lag_1_Freq']

    # K. PREDIKSI (Hanya memanggil fitur yang ada di mapping)
    X_f = pd.DataFrame([last_row[feature_mapping['Claim_Frequency']]])
    pred_f = max(1, final_models['Claim_Frequency'].predict(X_f)[0])

    X_s = pd.DataFrame([last_row[feature_mapping['Claim_Severity']]])
    pred_s = max(1e-6, final_models['Claim_Severity'].predict(X_s)[0])

    pred_t = pred_f * pred_s

    # L. Finalisasi Baris
    new_entry = last_row.copy()
    new_entry['Claim_Frequency'], new_entry['Claim_Severity'], new_entry['Total_Claim'] = pred_f, pred_s, pred_t
    current_df = pd.concat([current_df, pd.DataFrame([new_entry])], ignore_index=True)
    
    # M. Simpan (Hanya simpan yang dibutuhkan)
    m_id = f"{year}_{month:02d}"
    submission_data.append({'id': m_id, 'Freq': pred_f, 'Sev': pred_s, 'Total': pred_t})

print("Forecasting 2025-2026 selesai dengan imputasi cuaca otomatis.")

In [ ]:

df_submission_final = pd.DataFrame(submission_data)
filename = 'Tambahan/prediksi_2026.csv'

df_submission_final.to_csv(filename, index=False)

print(f"File berhasil disimpan: {filename}")
print("Preview hasil akhir:")
display(df_submission_final)

In [ ]:

# 1. Baca file yang baru saja disimpan
file_path = 'Tambahan/prediksi_2026.csv'
df = pd.read_csv(file_path)

# 2. Ubah nama kolom agar sesuai dengan label yang diminta (Frequency, Severity, Total)
df = df.rename(columns={
    'Freq': 'Claim_Frequency',
    'Sev': 'Claim_Severity',
    'Total': 'Total_Claim'
})

# 3. Gunakan melt untuk mengubah kolom menjadi baris
# 'id' tetap sebagai pengidentifikasi waktu (2025_08, dst)
df_melted = df.melt(id_vars='id', 
                    value_vars=['Claim_Frequency', 'Claim_Severity', 'Total_Claim'],
                    var_name='Metric', 
                    value_name='value')

# 4. Buat kolom 'id' baru dengan menggabungkan waktu dan nama metrik
df_melted['new_id'] = df_melted['id'] + "_" + df_melted['Metric']

# 5. Pilih kolom yang dibutuhkan dan urutkan agar rapi per periode
df_final = df_melted[['new_id', 'value']].rename(columns={'new_id': 'id'})
df_final = df_final.sort_values('id').reset_index(drop=True)

# 6. Simpan hasil akhir ke CSV baru
output_path = 'Tambahan/prediksi_2026.csv'
df_final.to_csv(output_path, index=False)

# Tampilkan hasil
print(f"Transformasi selesai! File disimpan di: {output_path}")
print(df_final.head(6))

# Evaluation

## Submission Graph

In [ ]:
# 1. Proses Data Prediksi 2025 (Script B)
df_pred = pd.read_csv('Tambahan/submission_kaggle.csv')
df_pred['Month_Num'] = df_pred['id'].apply(lambda x: int(x.split('_')[1]))
df_pred['Target'] = df_pred['id'].apply(lambda x: "_".join(x.split('_')[2:]))
df_pred_pivot = df_pred.pivot(index='Month_Num', columns='Target', values='value').reset_index()
df_pred_pivot['Version'] = '2025 (Predicted)'

# 2. Proses Data Aktual 2024 dari monthly_data_clean
# Pastikan monthly_data_clean memiliki kolom Year_Num dan Month_Num
df_actual_2024 = monthly_data_clean[monthly_data_clean['Year_Num'] == 2024].copy()

# Ambil hanya bulan yang ada di hasil prediksi (misal: 8, 9, 10, 11, 12)
target_months = df_pred_pivot['Month_Num'].unique()
df_actual_2024 = df_actual_2024[df_actual_2024['Month_Num'].isin(target_months)]

# Pilih kolom yang relevan dan beri label versi
cols_to_keep = ['Month_Num', 'Total_Claim', 'Claim_Frequency', 'Claim_Severity']
df_actual_2024 = df_actual_2024[cols_to_keep]
df_actual_2024['Version'] = '2024 (Actual)'

# 3. Gabungkan Data untuk Perbandingan
df_final_compare = pd.concat([df_actual_2024, df_pred_pivot], ignore_index=True)

# 4. Visualisasi Perbandingan YoY (Year-over-Year)
plt.style.use('seaborn-v0_8-whitegrid')
targets = ['Total_Claim', 'Claim_Frequency', 'Claim_Severity']
fig, axes = plt.subplots(len(targets), 1, figsize=(14, 5 * len(targets)), sharex=True)

month_labels = {8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'}

for i, target in enumerate(targets):
    sns.lineplot(data=df_final_compare, x='Month_Num', y=target, hue='Version', 
                 marker='o', ax=axes[i], linewidth=3, palette=['#2ecc71', '#e74c3c'])
    
    # Tambahkan Anotasi Angka
    for ver in df_final_compare['Version'].unique():
        subset = df_final_compare[df_final_compare['Version'] == ver]
        for x, y in zip(subset['Month_Num'], subset[target]):
            axes[i].text(x, y, f'{y:,.0f}', 
                         va='bottom', ha='center', fontsize=9, fontweight='bold',
                         bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))

    axes[i].set_title(f'YoY Comparison (2024 vs 2025): {target.replace("_", " ")}', fontsize=15, fontweight='bold')
    axes[i].set_ylabel('Value')
    axes[i].grid(True, alpha=0.3)

plt.xticks(target_months, [month_labels.get(m, m) for m in target_months])
plt.xlabel('Month (Comparison of Same Period)')
plt.tight_layout()
plt.show()

# YOY Comparison Forecast 2026

In [ ]:
# 1. AMBIL DATA AKTUAL (Dari monthly_data: Jan 2024 - Juli 2025)
cols_needed = ['Year_Num', 'Month_Num', 'Total_Claim', 'Claim_Frequency', 'Claim_Severity']
df_actual = monthly_data[
    ((monthly_data['Year_Num'] == 2024)) | 
    ((monthly_data['Year_Num'] == 2025) & (monthly_data['Month_Num'] <= 7))
][cols_needed].copy()
df_actual['Type'] = 'Actual'

# 2. AMBIL DATA PREDIKSI (Dari file script 74: Ags 2025 - Des 2026)
df_pred_raw = pd.read_csv('Tambahan/prediksi_2026.csv')
df_pred_raw['Year_Num'] = df_pred_raw['id'].apply(lambda x: int(x.split('_')[0]))
df_pred_raw['Month_Num'] = df_pred_raw['id'].apply(lambda x: int(x.split('_')[1]))
df_pred_raw['Target'] = df_pred_raw['id'].apply(lambda x: "_".join(x.split('_')[2:]))

df_pred = df_pred_raw.pivot(index=['Year_Num', 'Month_Num'], columns='Target', values='value').reset_index()
df_pred['Type'] = 'Forecast'

# 3. GABUNGKAN SEMUA
df_full_timeline = pd.concat([df_actual, df_pred], ignore_index=True)

# 4. VISUALISASI (Dipisah menjadi 3 Grafik)
plt.style.use('seaborn-v0_8-whitegrid')
targets = ['Total_Claim', 'Claim_Frequency', 'Claim_Severity']
palette = {2024: '#2ecc71', 2025: '#f1c40f', 2026: '#e74c3c'} # Hijau, Kuning, Merah

for target in targets:
    plt.figure(figsize=(14, 6))
    
    for year in [2024, 2025, 2026]:
        subset = df_full_timeline[df_full_timeline['Year_Num'] == year].sort_values('Month_Num')
        
        if year == 2025:
            # Bagian Actual 2025 (Jan-Jul) - Garis Tebal
            actual_part = subset[subset['Type'] == 'Actual']
            sns.lineplot(data=actual_part, x='Month_Num', y=target, 
                         color=palette[year], label='2025 (Actual)', marker='o', linewidth=3)
            
            # Bagian Forecast 2025 (Jul-Des) - Garis Putus-putus
            # Mulai dari Juli agar garis tersambung
            forecast_part = subset[subset['Month_Num'] >= 7]
            sns.lineplot(data=forecast_part, x='Month_Num', y=target, 
                         color=palette[year], label='2025 (Forecast)', linestyle='--', marker='X', linewidth=3)
        else:
            # Tahun 2024 dan 2026
            t_label = 'Actual' if year == 2024 else 'Forecast'
            sns.lineplot(data=subset, x='Month_Num', y=target, 
                         color=palette[year], label=f'{year} ({t_label})', marker='o', linewidth=2.5)

    # Pengaturan Judul dan Label
    plt.title(f'Analisis YoY: {target.replace("_", " ")} (2024 - 2026)', fontsize=15, fontweight='bold', pad=15)
    plt.xlabel('Bulan', fontsize=12)
    plt.ylabel('Nilai Prediksi', fontsize=12)
    
    # Sumbu X menggunakan nama bulan
    plt.xticks(range(1, 13), [calendar.month_name[i][:3] for i in range(1, 13)])
    
    plt.legend(title="Status Data", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Menampilkan grafik satu per satu
    plt.show()